# Introduction to the extended version of DiCE (Diverse Counterfactual Explanations)

[Mothilal et al. (2020)](https://dl.acm.org/doi/10.1145/3351095.3372850) introduce their method of generating counterfactual explanations considering _feasibility_, and _diversity_. [Guidotti and Ruggieri (2021)](https://link.springer.com/chapter/10.1007/978-3-030-88942-5_28), claim counterfactual explanations to be robust they should be similar for similar instances when they explain. In this study, in a search to improve the quality and reliability of the counterfactual explanations _robustness_ is found to be helpful and it also introduced in the optimization function.

DiCE-Extended is built upon the [DiCE (Diverse Counterfactual Explanations)](https://github.com/interpretml/DiCE) [(Mothilal et al. 2020)](https://dl.acm.org/doi/10.1145/3351095.3372850) framework by introducing a robustness term in the optimization function.

## Manipulated Optimization Function

The core enhancement in DiCE-Extended is the manipulated optimization function, designed to balance proximity, diversity, and feasibility of counterfactuals. The function is formulated as:

<a id="equation-1"></a>
\begin{equation}
\tag{1}
C(x) = \underset{c_1, ..., c_k}{\text{arg min}}
\frac{1}{2} \sum_{i=1}^{k} yloss(f(c_i), y) +
\frac{\lambda_1}{k} \sum_{i=1}^{k} dist(c_i, x) -
\lambda_2 \cdot dpp\_diversity(c_1, ..., c_k) -
\frac{\lambda_3}{k} \sum_{i=1}^{k} robustness(c_i, c_i')
\end{equation}

- **Proximity Loss**: The first term that averages the distance between generated counterfactuals and the original input ensure the counterfactuals to be as close as possible to the original input.
- **Diversity Loss**: Diversity of the counterfactual explanations is aquired by determinental point process of which loss is represented by the second term and it ensures that _k_ number of counterfactual explanations are generated.
- **Robustness Loss**: [Guidotti (2024)](https://link.springer.com/article/10.1007/s10618-022-00831-6) defines robustness as necessity of similar instances being explained by similar counterfactual explanations such that if $b(x_1)=b(x_2)=y$ then an explainer $f$ should generate counterfactuals $c_1$ and $c_2$ that are similar and can explain $x_1$ and $x_2$. The robustness term that is based on [Dice-Sørensen Coefficient](https://en.wikipedia.org/wiki/Dice-S%C3%B8rensen_coefficient), is adopted from [Bonasera and Carrizosa (2024)](
https://doi.org/10.48550/arXiv.2407.00843).

\begin{equation}
\tag{2}
Robustness(c_i, c_i') = \frac{2 * \lvert c_i \cap c_i' \rvert}{\lvert c_i \rvert + \lvert c_i' \rvert}
\end{equation}


By adjusting the weights $\lambda_1$, $\lambda_2$, $\lambda_3$ counterfactual explanations can be customised by specific needs.

### IMPORTANT NOTE

Some of the calculations in this notebook may yield slightly different results across runs. This variability is due to the stochastic nature of optimization processes, random initializations, and other computation dependent factors. Please keep this in mind when interpreting the results. For reproducibility, consider setting random seeds where applicable.

In [2]:
import sys
dice_path = "/Users/volk/Documents/bau24-25/thesis/repos/DiCE-X"
sys.path.insert(0, dice_path)

In [2]:
import dice_ml_x
from dice_ml_x.utils import helpers, neuralnetworks
import pickle
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
import torch
import numpy as np
import itertools
from tqdm import tqdm
import pandas as pd
import random
import tensorflow as tf
import os
from collections import OrderedDict

In [3]:
%load_ext autoreload
%autoreload 2

In [20]:
all_datasets = {
    "compas-recidivism": {
        "data": helpers.load_compas_dataset(),
        "target": "twoyearrecid"
    },
    "adult-income": {
        "data": helpers.load_adult_income_dataset(),
        "target": "income"
    },
    "lending-club": {
        "data": helpers.load_lending_club_dataset(),
        "target": "loan_status"
    },
    "german-credit": {
        "data": helpers.load_german_credit_dataset(),
        "target": "credit_risk"
    }
}

for datasetname, info in all_datasets.items():
    print(datasetname)
    print("number of examples -> ", len(info["data"]))
    print("number of features -> ", len(info["data"].columns))

compas-recidivism
number of examples ->  4966
number of features ->  6
adult-income
number of examples ->  32561
number of features ->  9
lending-club
number of examples ->  39715
number of features ->  9
german-credit
number of examples ->  1000
number of features ->  21


In [4]:
from dice_ml_x.utils import helpers
l_c_data = helpers.load_lending_club_dataset()
g_c_data = helpers.load_german_credit_dataset()

print(l_c_data.dtypes)
print(g_c_data.describe())

employment_years         int64
num_open_credit_acc      int64
annual_income          float64
loan_grade              object
credit_history         float64
purpose                 object
home                    object
addr_state              object
loan_status              int64
dtype: object
       duration_in_month  credit_amount  \
count        1000.000000    1000.000000   
mean           20.903000    3271.258000   
std            12.058814    2822.736876   
min             4.000000     250.000000   
25%            12.000000    1365.500000   
50%            18.000000    2319.500000   
75%            24.000000    3972.250000   
max            72.000000   18424.000000   

       installment_rate_in_percentage_of_disposable_income  \
count                                        1000.000000     
mean                                            2.973000     
std                                             1.118715     
min                                             1.000000     
25%      

In [21]:
from dice_ml_x.benchmarking import Benchmarking
datasets = [(helpers.load_compas_dataset(), "twoyearrecid", "compas-recidivism"),
            (helpers.load_adult_income_dataset(), "income", "adult-income"),
             (helpers.load_lending_club_dataset(), "loan_status", "lending-club"),
             (helpers.load_german_credit_dataset(), "credit_risk", "german-credit")]
backends = ["sklearn", "PYT", "TF2"]
benchmarking = Benchmarking(datasets=datasets,
                            backends=backends)
benchmarking.load_and_train(batch_size=16)

Benchmarking:   0%|          | 0/12 [00:00<?, ?it/s]


ValueError: too many values to unpack (expected 6)

### Explainers' Loss Chart

During the counterfactual generation process we keep track of each losses computed for the optimization process. The chart below shows that the counterfactuals are optimized depending on various type of losses i.e., class loss, proximity loss, diversity loss, and robustness loss. The algorithm although reached a plateau around 50th iteration since the stopping criteria which is loss difference isn't met the loop keeps the calculations. The other stopping criteria is maximum number of iterations which is set to 5000. The chart is plotted without considering the weights for losses. 

In [4]:
with open('benchmarking_results_23_01_2025-01_05.pkl', 'rb') as res_file:
    benchmarking_results = pickle.load(res_file)

2025-06-29 18:09:18.456961: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4 Pro
2025-06-29 18:09:18.456991: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 48.00 GB
2025-06-29 18:09:18.456997: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 18.00 GB
2025-06-29 18:09:18.457029: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:303] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-06-29 18:09:18.457045: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:269] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [ ]:
orig_data_model_performance_rows = []
for dataset_name, info in benchmarking_results.items():
    for backend_name, backend_res_dict in info.items():
        metrics = backend_res_dict['model_metrics']
        row = {
            'dataset': dataset_name,
            'backend': backend_name,
            'accuracy': round(metrics['accuracy'], 2),
            'f1_score': round(metrics['f1_score'], 2),
            'recall': round(metrics['recall'], 2),
            'precision': round(metrics['precision'], 2),
            'auc': round(metrics['auc'], 2)
        }
        orig_data_model_performance_rows.append(row)

orig_data_model_perf_df = pd.DataFrame(orig_data_model_performance_rows)
print(orig_data_model_perf_df)

    

In [ ]:
import matplotlib.pyplot as plt

def plot_loss_metrics_grid(benchmarking_results, datasets, backends,
                           proximity_weight=1.0, diversity_weight=1.0, robustness_weight=1.0,
                           save=False, fig_file_name=''):
    num_datasets = len(datasets)
    num_backends = len(backends)

    fig, axes = plt.subplots(num_datasets, num_backends, figsize=(15, 12), constrained_layout=True)
    fig.suptitle("Loss Metrics Across Datasets and Models", fontsize=16, y=1.02)

    for i, dataset in enumerate(datasets):
        for j, backend in enumerate(backends):
            ax = axes[i, j]
            
            if backend in benchmarking_results[dataset]:
                loss_metrics = benchmarking_results[dataset][backend]['exp_history']

                proximity_loss = loss_metrics["proximity_loss"] if proximity_weight == 1.0 else [
                    loss * proximity_weight for loss in loss_metrics["proximity_loss"]
                ]

                robustness_loss = loss_metrics["robustness_loss"] if robustness_weight == 1.0 else [
                    loss * robustness_weight for loss in loss_metrics["robustness_loss"]
                ]
                if backend != 'sklearn':
                    diversity_loss = loss_metrics["diversity_loss"] if diversity_weight == 1.0 else [
                        loss * diversity_weight for loss in loss_metrics["diversity_loss"]
                    ]
                yloss = loss_metrics["y_loss"]

                ax.plot(proximity_loss, label="Proximity Loss", marker='o')
                if backend != 'sklearn':
                    ax.plot(diversity_loss, label="Diversity Loss", marker='o')
                ax.plot(yloss, label="Y Loss", marker='^')

                ax.plot(robustness_loss, label="Robustness Loss", marker='v')

            ax.set_title(backend)
            ax.set_xlabel("Index (Iterations)")
            ax.set_ylabel(dataset)
            ax.legend()
            ax.grid()

    
    if save:
        root_dir = 'figure_artefacts'
        if not os.path.exists(root_dir):
            os.makedirs(root_dir)
        fig_file_path = os.path.join(root_dir, fig_file_name)
        plt.savefig(fig_file_path, format='eps')
    plt.show()
datasets = ['adult-income', 'lending-club', 'compas-recidivism', 'german-credit']
backends = ['PYT', 'TF2', 'sklearn']
plot_loss_metrics_grid(benchmarking_results, datasets, backends, save=True, fig_file_name='explainer_loss.eps')

In [ ]:
import matplotlib.pyplot as plt
import os

def plot_loss_metrics_2x2_for_pyt(benchmarking_results, datasets,
                                  proximity_weight=1.0, diversity_weight=1.0, robustness_weight=1.0,
                                  save=False, fig_file_name=''):

    fig, axes = plt.subplots(2, 2, figsize=(12, 10), constrained_layout=True)
    axes = axes.flatten() 

    legend_lines = []
    legend_labels = []

    for i, dataset in enumerate(datasets):
        if i >= 4:
            break 

        ax = axes[i]
        ax.set_title(f"{dataset}")
        ax.set_xlabel("Iterations")
        ax.set_ylabel("Loss Value")
        ax.grid(True)

        if dataset not in benchmarking_results:
            continue
        if 'TF2' not in benchmarking_results[dataset]:
            continue
        if 'exp_history' not in benchmarking_results[dataset]['TF2']:
            continue

        loss_metrics = benchmarking_results[dataset]['TF2']['exp_history']

        proximity_loss = loss_metrics["proximity_loss"][:150]
        robustness_loss = loss_metrics["robustness_loss"][:150]
        yloss = loss_metrics["y_loss"][:150]
        diversity_loss = loss_metrics["diversity_loss"][:150]

        if proximity_weight != 1.0:
            proximity_loss = [val * proximity_weight for val in proximity_loss]
        if diversity_weight != 1.0:
            diversity_loss = [val * diversity_weight for val in diversity_loss]
        if robustness_weight != 1.0:
            robustness_loss = [val * robustness_weight for val in robustness_loss]

        line1 = ax.plot(proximity_loss,
                        label="Proximity", marker='o',
                        markevery=5, markersize=4)
        line2 = ax.plot(diversity_loss, label="Diversity", marker='x',
                        markevery=5, markersize=4)
        line3 = ax.plot(yloss, label="Y-Loss", marker='^',
                        markevery=5, markersize=4)
        line4 = ax.plot(robustness_loss, label="Robustness", marker='v',
                        markevery=5, markersize=4)

        if i == 0:
            legend_lines.extend(line1 + line2 + line3 + line4)

    for j in range(i+1, 4):
        axes[j].set_visible(False)


    fig.legend(legend_lines, [l.get_label() for l in legend_lines],
               loc='lower center', ncol=4, bbox_to_anchor=(0.5, -0.05))

    if save:
        root_dir = 'figure_artefacts'
        if not os.path.exists(root_dir):
            os.makedirs(root_dir)
        fig_file_path = os.path.join(root_dir, fig_file_name)
        plt.savefig(fig_file_path, format='eps', bbox_inches="tight")

    plt.show()

datasets = ['adult-income', 'lending-club', 'compas-recidivism', 'german-credit']
plot_loss_metrics_2x2_for_pyt(benchmarking_results, datasets,
                              save=True, fig_file_name='explainer_loss_tf2_dice_x.eps')


In [ ]:
import matplotlib.pyplot as plt
import os
import numpy as np

def normalize(arr):
    """Normalize a 1D array to the range [0,1]."""
    arr = np.array(arr)
    if arr.size == 0:
        return arr
    min_val = arr.min()
    max_val = arr.max()
    if max_val - min_val == 0:
        return np.zeros_like(arr)
    return (arr - min_val) / (max_val - min_val)

def plot_loss_metrics_2x2_for_pyt(benchmarking_results, datasets,
                                  proximity_weight=1.0, diversity_weight=1.0, robustness_weight=1.0,
                                  save=False, fig_file_name=''):
    # Create a 2x2 grid for the selected datasets.
    fig, axes = plt.subplots(2, 2, figsize=(12, 10), constrained_layout=True)
    axes = axes.flatten()

    legend_lines = []

    for i, dataset in enumerate(datasets[:4]):
        ax = axes[i]
        ax.set_title(f"{dataset}")
        ax.set_xlabel("Iterations")
        ax.set_ylabel("Normalized Loss Value")
        ax.grid(True)

        # Remove y-tick labels for subplots in the right column (i.e., indices 1 and 3)
        if i % 2 == 1:
            ax.tick_params(labelleft=False)

        # Check if the required keys exist in benchmarking_results
        if dataset not in benchmarking_results:
            print(f"Dataset {dataset} not found in results!")
            continue
        if 'TF2' not in benchmarking_results[dataset]:
            print(f"TF2 backend not found for dataset {dataset}!")
            continue
        if 'exp_history' not in benchmarking_results[dataset]['TF2']:
            print(f"exp_history not found for dataset {dataset} under TF2!")
            continue

        loss_metrics = benchmarking_results[dataset]['TF2']['exp_history']

        # Use only the first 150 data points for each loss
        prox_loss = loss_metrics["proximity_loss"][:150]
        divers_loss = loss_metrics["diversity_loss"][:150]
        y_loss = loss_metrics["y_loss"][:150]
        robust_loss = loss_metrics["robustness_loss"][:150]

        if len(prox_loss) == 0 or len(y_loss) == 0:
            print(f"No loss data available for dataset {dataset}.")
            continue

        # Apply weight multipliers if needed
        if proximity_weight != 1.0:
            prox_loss = [val * proximity_weight for val in prox_loss]
        if diversity_weight != 1.0:
            divers_loss = [val * diversity_weight for val in divers_loss]
        if robustness_weight != 1.0:
            robust_loss = [val * robustness_weight for val in robust_loss]

        # Normalize each loss array to [0,1]
        prox_norm = normalize(prox_loss)
        divers_norm = normalize(divers_loss)
        y_norm = normalize(y_loss)
        robust_norm = normalize(robust_loss)

        # Plot the losses with markers plotted every 5 points, and smaller markers.
        line1 = ax.plot(prox_norm, label="Proximity", marker='o', markevery=5, markersize=4)
        line2 = ax.plot(divers_norm, label="Diversity", marker='x', markevery=5, markersize=4)
        line3 = ax.plot(y_norm, label="Y-Loss", marker='^', markevery=5, markersize=4)
        line4 = ax.plot(robust_norm, label="Robustness", marker='v', markevery=5, markersize=4)

        if i == 0:
            legend_lines.extend(line1 + line2 + line3 + line4)

    # Hide any extra subplots (if less than 4 datasets)
    for j in range(i+1, 4):
        axes[j].set_visible(False)

    # Create a single figure-level legend
    if legend_lines:
        fig.legend(legend_lines, [l.get_label() for l in legend_lines],
                   loc='lower center', ncol=4, bbox_to_anchor=(0.5, -0.05))

    if save:
        root_dir = 'figure_artefacts'
        if not os.path.exists(root_dir):
            os.makedirs(root_dir)
        fig_file_path = os.path.join(root_dir, fig_file_name)
        plt.savefig(fig_file_path, format='eps')
    
    plt.show()

# Example usage:
datasets = ['adult-income', 'lending-club', 'compas-recidivism', 'german-credit']
plot_loss_metrics_2x2_for_pyt(benchmarking_results, datasets,
                              save=True, fig_file_name='explainer_loss_tf2_normalized_dice_x.eps')


In [ ]:
import matplotlib.pyplot as plt
import os
import numpy as np

def log_normalize(arr, eps=1e-8):
    """
    Apply a log transform then normalize the result to the [0,1] range.
    The epsilon is added to avoid log(0) and to ensure values are strictly positive.
    """
    arr = np.array(arr)
    log_arr = np.log(arr + eps)
    # Normalize to [0,1]
    norm = (log_arr - log_arr.min()) / (log_arr.max() - log_arr.min() + eps)
    # Shift slightly to avoid exact zero values
    return norm + eps

def plot_loss_metrics_semilogy_lognorm_for_pyt(benchmarking_results, datasets,
                                               proximity_weight=1.0, diversity_weight=1.0, robustness_weight=1.0,
                                               save=False, fig_file_name=''):
    # Create a 2x2 grid for the selected datasets.
    fig, axes = plt.subplots(2, 2, figsize=(12, 10), constrained_layout=True)
    axes = axes.flatten()
    
    legend_lines = []
    eps = 1e-8  # epsilon to avoid log(0)
    
    for i, dataset in enumerate(datasets[:4]):
        ax = axes[i]
        ax.set_title(f"{dataset}")
        ax.set_xlabel("Iterations")
        ax.set_ylabel("Log-Normalized Loss")
        ax.grid(True, which="both")
        
        # Only show y-axis labels on left subplots (indices 0 and 2)
        if i % 2 == 1:
            ax.tick_params(labelleft=False)
        
        # Check for required keys.
        if dataset not in benchmarking_results:
            print(f"Dataset {dataset} not found in results!")
            continue
        if 'TF2' not in benchmarking_results[dataset]:
            print(f"TF2 backend not found for dataset {dataset}!")
            continue
        if 'exp_history' not in benchmarking_results[dataset]['TF2']:
            print(f"exp_history not found for dataset {dataset} under TF2!")
            continue
        
        loss_metrics = benchmarking_results[dataset]['TF2']['exp_history']
        
        # Use only the first 150 data points.
        prox_loss = np.array(loss_metrics["proximity_loss"][:150])
        divers_loss = np.array(loss_metrics["diversity_loss"][:150])
        y_loss = np.array(loss_metrics["y_loss"][:150])
        robust_loss = np.array(loss_metrics["robustness_loss"][:150])
        
        if len(prox_loss) == 0 or len(y_loss) == 0:
            print(f"No loss data available for dataset {dataset}.")
            continue
        
        # Apply weight multipliers if needed.
        prox_loss = prox_loss * proximity_weight
        divers_loss = divers_loss * diversity_weight
        robust_loss = robust_loss * robustness_weight
        
        iterations = np.arange(len(prox_loss))
        
        # Apply log transform and then normalize each loss array.
        prox_log_norm = log_normalize(prox_loss, eps)
        divers_log_norm = log_normalize(divers_loss, eps)
        y_log_norm = log_normalize(y_loss, eps)
        robust_log_norm = log_normalize(robust_loss, eps)
        
        # Plot using semilogy so that the y-axis is logarithmic.
        line1 = ax.semilogy(iterations, prox_log_norm, label="Proximity", 
                            marker='o', markevery=5, markersize=4)
        line2 = ax.semilogy(iterations, divers_log_norm, label="Diversity", 
                            marker='x', markevery=5, markersize=4)
        line3 = ax.semilogy(iterations, y_log_norm, label="Y-Loss", 
                            marker='^', markevery=5, markersize=4)
        line4 = ax.semilogy(iterations, robust_log_norm, label="Robustness", 
                            marker='v', markevery=5, markersize=4)
        
        if i == 0:
            legend_lines.extend(line1 + line2 + line3 + line4)
    
    # Hide any extra subplots if there are fewer than 4 datasets.
    for j in range(i+1, 4):
        axes[j].set_visible(False)
    
    # Create a single figure-level legend.
    if legend_lines:
        fig.legend(legend_lines, [l.get_label() for l in legend_lines],
                   loc='lower center', ncol=4, bbox_to_anchor=(0.5, -0.05))
    
    if save:
        root_dir = 'figure_artefacts'
        if not os.path.exists(root_dir):
            os.makedirs(root_dir)
        fig_file_path = os.path.join(root_dir, fig_file_name)
        plt.savefig(fig_file_path, format='eps')
    
    plt.show()

# Example usage:
datasets = ['adult-income', 'lending-club', 'compas-recidivism', 'german-credit']
plot_loss_metrics_semilogy_lognorm_for_pyt(benchmarking_results, datasets,
                                           save=True, fig_file_name='explainer_loss_semilogy_lognorm_pyt_dice_x.eps')


In [ ]:
import matplotlib.pyplot as plt
import os
import numpy as np

def plot_loss_metrics_loglog_for_pyt(benchmarking_results, datasets,
                                      proximity_weight=1.0, diversity_weight=1.0, robustness_weight=1.0,
                                      save=False, fig_file_name=''):
    # Create a 2x2 grid for the selected datasets.
    fig, axes = plt.subplots(2, 2, figsize=(12, 10), constrained_layout=True)
    axes = axes.flatten()
    
    legend_lines = []
    eps = 1e-8  # small value to avoid log(0)
    
    for i, dataset in enumerate(datasets[:4]):
        ax = axes[i]
        ax.set_title(f"{dataset}")
        ax.set_xlabel("Iterations (log scale)")
        ax.set_ylabel("Loss Value (log scale)")
        ax.grid(True, which="both")
        
        # Remove y-axis tick labels on the right subplots.
        if i % 2 == 1:
            ax.tick_params(labelleft=False)
        
        # Check required keys exist.
        if dataset not in benchmarking_results:
            print(f"Dataset {dataset} not found in results!")
            continue
        if 'TF2' not in benchmarking_results[dataset]:
            print(f"TF2 backend not found for dataset {dataset}!")
            continue
        if 'exp_history' not in benchmarking_results[dataset]['TF2']:
            print(f"exp_history not found for dataset {dataset} under TF2!")
            continue
        
        loss_metrics = benchmarking_results[dataset]['TF2']['exp_history']
        
        # Take only the first 150 iterations
        prox_loss = np.array(loss_metrics["proximity_loss"][:150]) + eps
        divers_loss = np.array(loss_metrics["diversity_loss"][:150]) + eps
        y_loss = np.array(loss_metrics["y_loss"][:150]) + eps
        robust_loss = np.array(loss_metrics["robustness_loss"][:150]) + eps
        
        # Apply weight multipliers if needed.
        prox_loss = prox_loss * proximity_weight
        divers_loss = divers_loss * diversity_weight
        robust_loss = robust_loss * robustness_weight
        
        iterations = np.arange(len(prox_loss))
        
        # Use log-log plotting: both axes on log scale.
        line1 = ax.loglog(iterations, prox_loss, label="Proximity", marker='o', markevery=5, markersize=4)
        line2 = ax.loglog(iterations, divers_loss, label="Diversity", marker='x', markevery=5, markersize=4)
        line3 = ax.loglog(iterations, y_loss, label="Y-Loss", marker='^', markevery=5, markersize=4)
        line4 = ax.loglog(iterations, robust_loss, label="Robustness", marker='v', markevery=5, markersize=4)
        
        if i == 0:
            legend_lines.extend(line1 + line2 + line3 + line4)
    
    # Hide any unused subplots.
    for j in range(i+1, 4):
        axes[j].set_visible(False)
    
    # Create a single figure-level legend.
    if legend_lines:
        fig.legend(legend_lines, [l.get_label() for l in legend_lines],
                   loc='lower center', ncol=4, bbox_to_anchor=(0.5, -0.05))
    
    if save:
        root_dir = 'figure_artefacts'
        if not os.path.exists(root_dir):
            os.makedirs(root_dir)
        fig_file_path = os.path.join(root_dir, fig_file_name)
        plt.savefig(fig_file_path, format='eps')
    
    plt.show()

# Example usage:
datasets = ['adult-income', 'lending-club', 'compas-recidivism', 'german-credit']
plot_loss_metrics_loglog_for_pyt(benchmarking_results, datasets,
                                 save=True, fig_file_name='explainer_loss_loglog_pyt_dice_x.eps')


In [ ]:
import matplotlib.pyplot as plt
import os
import numpy as np

def log_transform(arr, eps=1e-8):
    """Convert a 1D array to log scale (natural logarithm)."""
    arr = np.array(arr)
    return np.log(arr + eps)

def plot_loss_metrics_2x2_for_pyt(benchmarking_results, datasets,
                                  proximity_weight=1.0, diversity_weight=1.0, robustness_weight=1.0,
                                  save=False, fig_file_name=''):
    # Create a 2x2 grid for the selected datasets.
    fig, axes = plt.subplots(2, 2, figsize=(12, 10), constrained_layout=True)
    axes = axes.flatten()

    legend_lines = []

    for i, dataset in enumerate(datasets[:4]):
        ax = axes[i]
        ax.set_title(f"{dataset}")
        ax.set_xlabel("Iterations")
        ax.set_ylabel("Log(Loss Value)")
        ax.grid(True)

        # Remove y-tick labels for subplots in the right column (indices 1 and 3)
        if i % 2 == 1:
            ax.tick_params(labelleft=False)

        # Check if the required keys exist in benchmarking_results
        if dataset not in benchmarking_results:
            print(f"Dataset {dataset} not found in results!")
            continue
        if 'TF2' not in benchmarking_results[dataset]:
            print(f"TF2 backend not found for dataset {dataset}!")
            continue
        if 'exp_history' not in benchmarking_results[dataset]['TF2']:
            print(f"exp_history not found for dataset {dataset} under TF2!")
            continue

        loss_metrics = benchmarking_results[dataset]['TF2']['exp_history']

        # Use only the first 150 data points for each loss.
        prox_loss = loss_metrics["proximity_loss"][:150]
        divers_loss = loss_metrics["diversity_loss"][:150]
        y_loss = loss_metrics["y_loss"][:150]
        robust_loss = loss_metrics["robustness_loss"][:150]

        if len(prox_loss) == 0 or len(y_loss) == 0:
            print(f"No loss data available for dataset {dataset}.")
            continue

        # Apply weight multipliers if needed.
        if proximity_weight != 1.0:
            prox_loss = [val * proximity_weight for val in prox_loss]
        if diversity_weight != 1.0:
            divers_loss = [val * diversity_weight for val in divers_loss]
        if robustness_weight != 1.0:
            robust_loss = [val * robustness_weight for val in robust_loss]

        # Apply logarithmic transformation to each loss array.
        prox_log = log_transform(prox_loss)
        divers_log = log_transform(divers_loss)
        y_log = log_transform(y_loss)
        robust_log = log_transform(robust_loss)

        # Plot the losses with markers (every 5 points) and reduced marker size.
        line1 = ax.plot(prox_log, label="Proximity", marker='o', markevery=5, markersize=4)
        line2 = ax.plot(divers_log, label="Diversity", marker='x', markevery=5, markersize=4)
        line3 = ax.plot(y_log, label="Y-Loss", marker='^', markevery=5, markersize=4)
        line4 = ax.plot(robust_log, label="Robustness", marker='v', markevery=5, markersize=4)

        if i == 0:
            legend_lines.extend(line1 + line2 + line3 + line4)

    # Hide any extra subplots if datasets < 4.
    for j in range(i+1, 4):
        axes[j].set_visible(False)

    # Create a single figure-level legend.
    if legend_lines:
        fig.legend(legend_lines, [l.get_label() for l in legend_lines],
                   loc='lower center', ncol=4, bbox_to_anchor=(0.5, -0.05))

    if save:
        root_dir = 'figure_artefacts'
        if not os.path.exists(root_dir):
            os.makedirs(root_dir)
        fig_file_path = os.path.join(root_dir, fig_file_name)
        plt.savefig(fig_file_path, format='eps')
    
    plt.show()

# Example usage:
datasets = ['adult-income', 'lending-club', 'compas-recidivism', 'german-credit']
plot_loss_metrics_2x2_for_pyt(benchmarking_results, datasets, save=True, fig_file_name='explainer_loss_pyt_dice_x.eps')


In [ ]:
import matplotlib.pyplot as plt
import os
import numpy as np

def log_normalize(arr, eps=1e-8):
    """
    Apply a natural log transform to a 1D array (adding eps to avoid log(0)) and 
    normalize the result to the [0,1] range.
    """
    arr = np.array(arr)
    log_arr = np.log(arr + eps)
    norm = (log_arr - log_arr.min()) / (log_arr.max() - log_arr.min() + eps)
    # Add eps again to avoid exact zero values when plotting on log scale
    return norm + eps

def plot_loss_metrics_loglog_lognorm_for_pyt(benchmarking_results, datasets,
                                             proximity_weight=1.0, diversity_weight=1.0, robustness_weight=1.0,
                                             save=False, fig_file_name=''):
    # Create a 2x2 grid for the selected datasets.
    fig, axes = plt.subplots(2, 2, figsize=(12, 10), constrained_layout=True)
    axes = axes.flatten()
    
    legend_lines = []
    eps = 1e-8  # Small epsilon to avoid log(0)
    
    # We'll use iterations+1 to avoid 0 on the x-axis for log-log plotting.
    iterations = np.arange(150) + 1  # First 150 iterations
    
    for i, dataset in enumerate(datasets):
        ax = axes[i]
        ax.set_title(f"{dataset}")
        ax.set_xlabel("Iterations (log scale)")
        ax.set_ylabel("Loss (log-normalized)")
        ax.grid(True, which="both")
        
        # Remove y-axis tick labels on subplots in the right column (indices 1 and 3)
        if i % 2 == 1:
            ax.tick_params(labelleft=False)
        
        # Check that the dataset has the required keys.
        if dataset not in benchmarking_results:
            print(f"Dataset {dataset} not found in results!")
            continue
        if 'TF2' not in benchmarking_results[dataset]:
            print(f"TF2 backend not found for dataset {dataset}!")
            continue
        if 'exp_history' not in benchmarking_results[dataset]['TF2']:
            print(f"exp_history not found for dataset {dataset} under TF2!")
            continue
        
        loss_metrics = benchmarking_results[dataset]['TF2']['exp_history']
        
        # Extract only the first 150 data points.
        prox_loss = np.array(loss_metrics["proximity_loss"][:150])
        divers_loss = np.array(loss_metrics["diversity_loss"][:150])
        y_loss = np.array(loss_metrics["y_loss"][:150])
        robust_loss = np.array(loss_metrics["robustness_loss"][:150])
        
        # Apply weight multipliers if needed.
        prox_loss = prox_loss * proximity_weight
        divers_loss = divers_loss * diversity_weight
        robust_loss = robust_loss * robustness_weight
        
        # Apply log transform then normalize each loss array.
        prox_log_norm = log_normalize(prox_loss, eps)
        divers_log_norm = log_normalize(divers_loss, eps)
        y_log_norm = log_normalize(y_loss, eps)
        robust_log_norm = log_normalize(robust_loss, eps)
        
        # Plot the losses using loglog: both axes in log scale.
        line1 = ax.loglog(iterations, prox_log_norm, label="Proximity", 
                           marker='o', markevery=5, markersize=4)
        line2 = ax.loglog(iterations, divers_log_norm, label="Diversity", 
                           marker='x', markevery=5, markersize=4)
        line3 = ax.loglog(iterations, y_log_norm, label="Y-Loss", 
                           marker='^', markevery=5, markersize=4)
        line4 = ax.loglog(iterations, robust_log_norm, label="Robustness", 
                           marker='v', markevery=5, markersize=4)
        
        if i == 0:
            legend_lines.extend(line1 + line2 + line3 + line4)
    
    # Hide any extra subplots if datasets < 4.
    for j in range(i+1, 4):
        axes[j].set_visible(False)
    
    # Create a single figure-level legend.
    if legend_lines:
        fig.legend(legend_lines, [l.get_label() for l in legend_lines],
                   loc='lower center', ncol=4, bbox_to_anchor=(0.5, -0.05))
    
    if save:
        root_dir = 'figure_artefacts'
        if not os.path.exists(root_dir):
            os.makedirs(root_dir)
        fig_file_path = os.path.join(root_dir, fig_file_name)
        plt.savefig(fig_file_path, format='eps')
    
    plt.show()

# Example usage:
datasets = ['adult-income', 'lending-club', 'compas-recidivism', 'german-credit']
plot_loss_metrics_loglog_lognorm_for_pyt(benchmarking_results, datasets,
                                         save=True, fig_file_name='explainer_loss_loglog_lognorm_pyt_dice_x.eps')


In [5]:
backends = ['sklearn', 'PYT', 'TF2']
def load_torch_model(model_path, in_features):
    dummy_state_dict = torch.load(model_path)
    dummy_state_dict = {f'model.{key}': value for key, value in dummy_state_dict.items()}
    model = neuralnetworks.PYTModel(in_features, model_save_dir="")
    model.load_state_dict(dummy_state_dict)
    return model


def load_tensorflow_model(model_path: str):
    model = neuralnetworks.TF2Model()
    model.load_weights(str(model_path))
    model.trainable = False
    return model

dataset_names = [
    "compas-recidivism",
    "adult-income",
    "lending-club",
    "german-credit"
]
models = {}
for name in dataset_names:
    models[name] = {}
    for backend in backends:
        if backend == 'sklearn':
            models[name][backend] = benchmarking_results[name][backend]['model']
        elif backend == 'PYT':
            model_path = benchmarking_results[name][backend]['model_path']
            num_features = benchmarking_results[name][backend]['metrics']['num_features']
            models[name][backend] = load_torch_model(model_path, num_features)
        elif backend == 'TF2':
            model_path = benchmarking_results[name][backend]['model_path']
            models[name][backend] = load_tensorflow_model(model_path)

In [ ]:
import pandas as pd
cf_ds_pyt_g_c = pd.read_csv("cfe_datasets/compas-recidivism_sklearn_cfe.csv")
total_dups = len(cf_ds_pyt_g_c.drop_duplicates())
total_dups_rate = 100.0 - round((total_dups / 1000.0) * 100.0, 2)
_0_cls_dups = len(cf_ds_pyt_g_c[cf_ds_pyt_g_c["twoyearrecid"] < 0.5].drop_duplicates())
_0_cls_dups_rate = 100.0 - round((_0_cls_dups / 500.0) * 100.0, 2)
_1_cls_dups = len(cf_ds_pyt_g_c[cf_ds_pyt_g_c["twoyearrecid"] >= 0.5].drop_duplicates())
_1_cls_dups_rate = 100.0 - round((_1_cls_dups / 500.0) * 100.0, 2)
print(f"Total unique -> number of rows: {total_dups}/1000, Duplicate rate: {total_dups_rate}%")
print(f"Class 0 unique -> number of rows: {_0_cls_dups}/500, Duplicate rate: {_0_cls_dups_rate}%")
print(f"Class 1 unique -> number of rows: {_1_cls_dups}/500, Duplicate rate: {_1_cls_dups_rate}%")

In [ ]:
datasets = [(helpers.load_compas_dataset(), "twoyearrecid", "compas-recidivism"),
            (helpers.load_adult_income_dataset(), "income", "adult-income"),
             (helpers.load_lending_club_dataset(), "loan_status", "lending-club"),
             (helpers.load_german_credit_dataset(), "credit_risk", "german-credit")]
backends = ['TF2']
root_folder = 'cfe_datasets_28_06_25'
os.environ["TQDM_DISABLE"] = "1"


cfe_datasets = {}


for exp_iteration in range(2):
    root_folder = f"{root_folder}_0{exp_iteration + 1}"
    os.makedirs(root_folder, exist_ok=True)
    pbar = tqdm(datasets, desc="Generating counterfactual datasets...")
    for df, target_name, df_name in pbar:
        target_col = df[target_name]
        train_dataset, test_dataset, y_train, y_test = train_test_split(df, target_col, test_size=0.2,
                                                                            random_state=42, stratify=target_col)
        cont_feats = df.select_dtypes(include=[np.number]).columns.difference([target_name])
        d = dice_ml_x.Data(dataframe=train_dataset, continuous_features=list(cont_feats), outcome_name=target_name)
        x_test = test_dataset.drop(columns=[target_name])
        

        for backend in backends:
            file_path = os.path.join(root_folder, f"{df_name}_{backend}_cfe.csv")

            if os.path.isfile(file_path):
                existing_df = pd.read_csv(file_path)
                num_rows = len(existing_df)
                if num_rows >= 1000:
                    print(f"Skipping {df_name} for the backend {backend} because it already exists and have {num_rows} rows.")
                    continue
            else:
                existing_df = pd.DataFrame()
                
            print(f"Processing {df_name} with {backend} ...")

            model_options = OrderedDict(model=models[df_name][backend], backend=backend)
            method = 'genetic' if backend == 'sklearn' else 'gradient'

            if backend != 'sklearn':
                model_options['func'] = 'ohe-min-max'
                if backend == 'PYT':
                    model_options['model'] = models[df_name][backend].model

            m = dice_ml_x.Model(**model_options)
            exp = dice_ml_x.Dice(d, m, method=method)

            if backend == 'PYT':
                x_test_ohe_vector = torch.tensor(d.get_ohe_min_max_normalized_data(x_test).values, dtype=torch.float32)
            elif backend == 'TF2':
                x_test_ohe_vector = tf.constant(d.get_ohe_min_max_normalized_data(x_test).values, dtype=tf.float32)
                preds = m.model.predict(x_test_ohe_vector)
                print(f"Prediction min/max/mean: {preds.min():.4f} / {preds.max():.4f} / {preds.mean():.4f}")
                print(f"Predicted class distribution: {(preds >= 0.5).sum()} class 1, {(preds < 0.5).sum()} class 0")
            '''if backend == 'PYT':
                _0_d_indices = np.where(m.model(x_test_ohe_vector) < 0.5)[0].tolist()
            elif backend == 'TF2':
                _0_d_indices = np.where(m.model.predict(x_test_ohe_vector) < 0.5)[0].tolist()
            else:
                _0_d_indices = np.where(m.model.predict(x_test) < 0.5)[0].tolist()

            _1_d_indices = list((set(range(len(x_test)))) - set(list(_0_d_indices)))'''

            y_test_values = y_test.values  # already aligned with x_test

            # Class-based indices (based on actual labels, not predictions)
            _0_d_indices = np.where(y_test_values < 0.5)[0].tolist()
            _1_d_indices = np.where(y_test_values >= 0.5)[0].tolist()


            print(f"\n[VERIFICATION] Dataset: {df_name}, Backend: {backend}")
            print(f"Found {len(_0_d_indices)} samples predicted as Class 0 in the test set.")
            print(f"Found {len(_1_d_indices)} samples predicted as Class 1 in the test set.\n")

            random.shuffle(_0_d_indices)
            random.shuffle(_1_d_indices)

            if not existing_df.empty:
                count_class_0 = (existing_df[target_name] < 0.5).sum()
                count_class_1 = (existing_df[target_name] >= 0.5).sum()
            else:
                count_class_0, count_class_1 = 0, 0
            
            target_count = 500
            
            cfe_dataset_list = []
    
            while count_class_0 < target_count or count_class_1 < target_count:
                

                '''if count_class_0 < target_count and _0_d_indices:
                    rand_idx = random.choice(_0_d_indices)
                    # do something with rand_idx
                elif count_class_1 < target_count and _1_d_indices:
                    rand_idx = random.choice(_1_d_indices)
                    # do something with rand_idx
                else:
                    print("No more available samples in one of the lists.")
                    continue'''
                
                if count_class_0 <= count_class_1:
                    desired_class = 0
                else:
                    desired_class = 1

                # If the chosen class is already full, switch to the other one.
                if (desired_class == 0 and count_class_0 >= target_count):
                    desired_class = 1
                elif (desired_class == 1 and count_class_1 >= target_count):
                    desired_class = 0

                # 2. Select a starting point from the OPPOSITE class.
                if desired_class == 0:
                    # To get a CFE of class 0, start with an instance predicted as class 1.
                    if not _1_d_indices:
                        # Handle case where source instances are missing
                        print("class 1 exists skipped the loop")
                        continue
                    rand_idx = random.choice(_1_d_indices)
                else: # desired_class == 1
                    # To get a CFE of class 1, start with an instance predicted as class 0.
                    if not _0_d_indices:
                        # Handle case where source instances are missing
                        print("class 0 exists skipped the loop") 
                        continue
                    rand_idx = random.choice(_0_d_indices)

                x = x_test[rand_idx : rand_idx + 1]

                '''if backend == 'PYT':
                    x_ohe = torch.tensor(d.get_ohe_min_max_normalized_data(x).values, dtype=torch.float32)
                    desired_class = int(m.model(x_ohe) >= 0.5)
                elif backend == 'TF2':
                    x_ohe = tf.constant(d.get_ohe_min_max_normalized_data(x).values, dtype=tf.float32)
                    desired_class = int(m.model.predict(x_ohe) >= 0.5)
                   
                else:
                    desired_class = int(m.model.predict(x) >= 0.5)

                desired_class = 1 - desired_class'''

                if (desired_class == 0 and count_class_0 >= target_count) or \
                (desired_class == 1 and count_class_1 >= target_count):
                    print("infinite loop is achieved")
                    continue

                total_cfs = 5
                dice_exp = exp.generate_counterfactuals(
                    x, total_CFs=total_cfs, desired_class=desired_class, robustness_weight=0.4,
                    max_iter=500, learning_rate=0.05
                )
                print(f"counterfactuals are generated for class {desired_class}")
                cfe_sample = dice_exp.to_dataframe()
                
                if desired_class == 0:
                    count_class_0 += total_cfs
                else:
                    count_class_1 += total_cfs
                
                
                cfe_dataset_list.append(cfe_sample)
                
                if len(cfe_dataset_list) % 2 == 0:
                    
                    combined_df = pd.concat(cfe_dataset_list, ignore_index=True).drop_duplicates()
                    
                    if os.path.isfile(file_path):
                        existing_df = pd.read_csv(file_path)
                        combined_df = pd.concat([combined_df, existing_df], ignore_index=True).drop_duplicates()
                    
                    count_class_0 = (combined_df[target_name] < 0.5).sum()
                    count_class_1 = (combined_df[target_name] >= 0.5).sum()

                    combined_df.to_csv(file_path, index=False)
                    
                    print(f"{df_name} - {backend}, {len(combined_df)} counterfactuals are saved to {file_path}")
                    cfe_dataset_list = []

                pbar.set_postfix(OrderedDict(backend=backend, dataset=df_name,
                                            sample_count_class_0=count_class_0,
                                            sample_count_class_1=count_class_1))
                pbar.update(1)
                '''except Exception as e:
                    print(f"Couldn't generate counterfactuals for {df_name}, {backend}")
                    continue'''
        

Generating counterfactual datasets...:   0%|          | 0/4 [00:00<?, ?it/s]

Skipping compas-recidivism for the backend TF2 because it already exists and have 1004 rows.
Skipping adult-income for the backend TF2 because it already exists and have 1004 rows.
Skipping lending-club for the backend TF2 because it already exists and have 1000 rows.
Processing german-credit with TF2 ...
7/7 [==============================] - 0s 1ms/step


Prediction min/max/mean: 0.0072 / 0.9342 / 0.4002
Predicted class distribution: 74 class 1, 126 class 0

[VERIFICATION] Dataset: german-credit, Backend: TF2
Found 140 samples predicted as Class 0 in the test set.
Found 60 samples predicted as Class 1 in the test set.



2025-06-29 19:50:32.529565: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.
Generating counterfactual datasets...: 100%|██████████| 4/4 [00:17<00:00,  4.37s/it, backend=TF2, dataset=german-credit, sample_count_class_0=503, sample_count_class_1=500]


Diverse Counterfactuals found! total time taken: 00 min 17 sec
counterfactuals are generated for class 0


...

Diverse Counterfactuals found! total time taken: 00 min 17 sec
counterfactuals are generated for class 0


Generating counterfactual datasets...: 934it [4:03:33, 18.24s/it, backend=TF2, dataset=german-credit, sample_count_class_0=496, sample_count_class_1=503]WARNING:root: MAD for feature number_of_existing_credits_at_this_bank is 0, so replacing it with 1.0 to avoid error.


Diverse Counterfactuals found! total time taken: 00 min 17 sec
counterfactuals are generated for class 1
german-credit - TF2, 999 counterfactuals are saved to cfe_datasets_28_06_25_01_02/german-credit_TF2_cfe.csv


Generating counterfactual datasets...: 100%|██████████| 4/4 [4:03:51<00:00, 3657.86s/it, backend=TF2, dataset=german-credit, sample_count_class_0=501, sample_count_class_1=503]

Diverse Counterfactuals found! total time taken: 00 min 17 sec
counterfactuals are generated for class 0


In [ ]:
tf.config.list_physical_devices("GPU")

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

In [ ]:
# Load datasets if the cell above stopped at some point
file_names = os.listdir('cfe_datasets')
keys = [(f_name.split('_')[0], f_name.split('_')[1]) for f_name in file_names]
target_names = {'lending-club': 'loan_status', 'compas-recidivism': 'twoyearrecid',
                'adult-income': 'income', 'german-credit': 'credit_risk'}
cfe_datasets = {}
for key, f_name in zip(keys, file_names):
    new_key = key + (target_names[key[0]],)
    cfe_datasets[new_key] = pd.read_csv(os.path.join('cfe_datasets', f_name))

In [ ]:
# Convert target column to binary (0 or 1 depending on whether its value is greater or less than 0.5)
for (df_name, backend_name, target_name), cfe_dataset in cfe_datasets.items():
    cfe_dataset[target_name] = np.where(cfe_dataset[target_name] < 0.5, 0, 1)

In [ ]:
# Train 12 models for each dataset and backend
all_datasets = {
    "compas-recidivism": {
        "data": helpers.load_compas_dataset(),
        "target": "twoyearrecid"
    },
    "adult-income": {
        "data": helpers.load_adult_income_dataset(),
        "target": "income"
    },
    "lending-club": {
        "data": helpers.load_lending_club_dataset(),
        "target": "loan_status"
    },
    "german-credit": {
        "data": helpers.load_german_credit_dataset(),
        "target": "credit_risk"
    }
}

from dice_ml_x.benchmarking import Benchmarking
artefact_path = 'cfe_models'
if not os.path.exists(artefact_path):
    os.makedirs(artefact_path)
cfe_model_results = {}
pbar = tqdm(cfe_datasets.items())
benchmarking = Benchmarking(None, None)
for (df_name, backend_name, target_name), cfe_dataset in pbar:
    cont_feats = cfe_dataset.select_dtypes(include=[np.number]).columns.difference([target_name])
    x_train_transformed, x_test_transformed, train_df, test_df, y_train, y_test, pipeline = benchmarking.preprocess_data(backend_name, cfe_dataset, cont_feats, target_name, 8)
    
    orig_df = all_datasets[df_name]['data']

    orig_target = all_datasets[df_name]['target']
    
    cat_cols = list(set(cfe_dataset.columns) - set(cont_feats))
    cat_col_unique_vals = {cat_col: cfe_dataset[cat_col].unique() for cat_col in cat_cols}
    for cl, unique_list in cat_col_unique_vals.items():
        orig_df_cat_unique = orig_df[cl].unique()

        if set(orig_df_cat_unique) != set(unique_list):
            orig_df = orig_df[orig_df[cl].isin(unique_list)]
            
    keep_cols = cfe_dataset.columns.tolist()
    orig_df = orig_df[keep_cols]
    orig_df.reset_index(drop=True, inplace=True)
    orig_cont_feats = orig_df.select_dtypes(include=[np.number]).columns.difference([orig_target]).tolist()
    
    _, orig_test_transformed, _, _, _, orig_y_test, _ = benchmarking.preprocess_data(backend_name, orig_df, orig_cont_feats, orig_target, 8, pipeline)
    model = benchmarking.train_model(backend_name, x_train_transformed, x_test_transformed, y_train)
    
    if backend_name == "sklearn":
        model_metrics = benchmarking.compute_RF_metrics(model, orig_test_transformed, orig_y_test)
    elif backend_name == "PYT":
        model_metrics = benchmarking.compute_pytorch_metrics(model, orig_test_transformed)
    elif backend_name == "TF2":
        model_metrics = benchmarking.compute_keras_metrics(model, orig_test_transformed, orig_y_test)

    key = (df_name, backend_name)
    cfe_model_results[key] = {
        'model_metrics': model_metrics
    }

    if backend_name == "PYT":
        model_path = os.path.join(artefact_path, f"{df_name}_{backend_name}_model.pth")
        torch.save(model.state_dict(), model_path)
        cfe_model_results[key]['model_path'] = model_path
    elif backend_name == "TF2":
        model_path = os.path.join(artefact_path, f"{df_name}_{backend_name}_model")
        model.save_weights(model_path, save_format='tf')
        cfe_model_results[key]['model_path'] = model_path
    else:
        model_path = os.path.join(artefact_path, f"{df_name}_{backend_name}_model.pkl")
        with open(model_path, 'wb') as model_file:
            pickle.dump(model, model_file)
        cfe_model_results[key]['model_path'] = model_path
    pbar.set_postfix(dataset=df_name, backend=backend_name)
    pbar.update(1)
    
model_result_path = os.path.join(artefact_path, 'cfe_model_metrics_result.pkl')
with open(model_result_path, 'wb') as model_res_file:
    pickle.dump(cfe_model_results, model_res_file)
    

In [ ]:
## Load the model metrics file generated by training models with counterfactuals datasets.
with open('cfe_models/model_metrics_result.pkl', 'rb') as cfe_model_metrics_file:
    cfe_model_results = pickle.load(cfe_model_metrics_file)

In [ ]:
rows = []
for (dataset, backend), info in cfe_model_results.items():
    metrics = info['model_metrics']
    row = {
        'dataset': dataset,
        'backend': backend,
        'accuracy': round(metrics['accuracy'], 2),
        'f1_score': round(metrics['f1_score'], 2),
        'recall': round(metrics['recall'], 2),
        'precision': round(metrics['precision'], 2),
        'auc': round(metrics['auc'], 2)
    }
    rows.append(row)

df = pd.DataFrame(rows)

backend_order = {'sklearn': 0, 'PYT': 1, 'TF2': 2}
df['backend_priority'] = df['backend'].map(backend_order)
df_sorted = df.sort_values(by=['dataset', 'backend_priority'])
df_sorted = df_sorted.drop(columns=['backend_priority'])
df_sorted = df_sorted.reset_index()
df_sorted.drop(columns="index", inplace=True)
print(df_sorted)

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier

cf_adult_df = cfe_datasets[("adult-income", "sklearn", "income")]
orig_adult_df = all_datasets["adult-income"]["data"]
orig_cont_feats = orig_adult_df.select_dtypes(include=[np.number]).columns.difference(["income"]).tolist()

train_df, test_df, y_train, y_test = train_test_split(orig_adult_df, orig_adult_df["income"], random_state=42,
                                                      shuffle=True, test_size=0.2, stratify=orig_adult_df["income"])
x_train, x_test = train_df.drop(columns=["income"]), test_df.drop(columns=["income"])
categorical = x_train.columns.difference(orig_cont_feats)

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))])

transformations = ColumnTransformer(
    transformers=[
        ('cat', categorical_transformer, categorical)])

sklearn_pipeline = Pipeline(steps=[('preprocessor', transformations),
                    ('classifier', RandomForestClassifier())])

sklearn_pipeline.fit(x_train, y_train)



cfe_df = cfe_datasets[("adult-income", "sklearn", "income")].drop(columns=["income"])
cfe_y_true = cfe_datasets[("adult-income", "sklearn", "income")]["income"]

accuracy_score(cfe_y_true, sklearn_pipeline.predict(cfe_df))


0.9556898288016112

In [ ]:
cf_compas_df = cfe_datasets[("compas-recidivism", "sklearn", "twoyearrecid")]
orig_compas_df = all_datasets["compas-recidivism"]["data"]
orig_cont_feats = orig_compas_df.select_dtypes(include=[np.number]).columns.difference(["twoyearrecid"]).tolist()

train_df, test_df, y_train, y_test = train_test_split(orig_compas_df, orig_compas_df["twoyearrecid"], random_state=42,
                                                      shuffle=True, test_size=0.2, stratify=orig_compas_df["twoyearrecid"])
x_train, x_test = train_df.drop(columns=["twoyearrecid"]), test_df.drop(columns=["twoyearrecid"])
categorical = x_train.columns.difference(orig_cont_feats)

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))])

transformations = ColumnTransformer(
    transformers=[
        ('cat', categorical_transformer, categorical)])

sklearn_pipeline = Pipeline(steps=[('preprocessor', transformations),
                    ('classifier', RandomForestClassifier())])

sklearn_pipeline.fit(x_train, y_train)

cfe_df = cfe_df.drop_duplicates()

cfe_df = cfe_datasets[("compas-recidivism", "sklearn", "twoyearrecid")].drop(columns=["twoyearrecid"])
cfe_y_true = cfe_datasets[("compas-recidivism", "sklearn", "twoyearrecid")]["twoyearrecid"]

accuracy_score(cfe_y_true, sklearn_pipeline.predict(cfe_df))

1.0

In [ ]:
###########################################################################
###########################################################################
###                                                                     ###
###                                                                     ###
###       Original models' performance on counterfactual datasets       ###
###                                                                     ###
###                                                                     ###
###########################################################################
###########################################################################


all_datasets = {
    "compas-recidivism": {
        "data": helpers.load_compas_dataset(),
        "target": "twoyearrecid"
    },
    "adult-income": {
        "data": helpers.load_adult_income_dataset(),
        "target": "income"
    },
    "lending-club": {
        "data": helpers.load_lending_club_dataset(),
        "target": "loan_status"
    },
    "german-credit": {
        "data": helpers.load_german_credit_dataset(),
        "target": "credit_risk"
    }
}

from dice_ml_x.benchmarking import Benchmarking

cfe_performance_results = {}
pbar = tqdm(cfe_datasets.items())
benchmarking = Benchmarking(None, None)
for (df_name, backend_name, target_name), cfe_dataset in pbar:

    orig_df = all_datasets[df_name]['data']
    orig_target = all_datasets[df_name]['target']
    orig_cont_feats = orig_df.select_dtypes(include=[np.number]).columns.difference([orig_target]).tolist()

    _, orig_test_transformed, _, orig_test_df, _, \
    orig_y_test, pipeline, pyt_scaler, pyt_encoder, pyt_label_encoder = benchmarking.preprocess_data(
        backend_name, orig_df, orig_cont_feats, orig_target, 8)

    cfe_cont_feats = cfe_dataset.select_dtypes(include=[np.number]).columns.difference([target_name])
    x_train_transformed, x_test_transformed, train_df, test_df, \
        y_train, y_test, _, _, _, _ = benchmarking.preprocess_data(backend_name, cfe_dataset,
                                                                 cfe_cont_feats, target_name, 8,
                                                                 pipeline, pyt_scaler=pyt_scaler,
                                                                 pyt_encoder=pyt_encoder,
                                                                 pyt_label_encoder=pyt_label_encoder,
                                                                 test_size=0.0)
    
    model = models[df_name][backend_name]
    
    if backend_name == "sklearn":
        model_metrics = benchmarking.compute_RF_metrics(model, x_train_transformed, y_train)
    elif backend_name == "PYT":
        model_metrics = benchmarking.compute_pytorch_metrics(model.model, x_train_transformed)
    elif backend_name == "TF2":
        model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
        model_metrics = benchmarking.compute_keras_metrics(model, x_train_transformed, y_train)

    key = (df_name, backend_name)
    cfe_performance_results[key] = {
        'model_metrics': model_metrics
    }

    if backend_name == "PYT":
        cfe_performance_results[key]['model_path'] = model_path
    elif backend_name == "TF2":
        cfe_performance_results[key]['model_path'] = model_path
    else:
        cfe_performance_results[key]['model_path'] = model_path
    pbar.set_postfix(dataset=df_name, backend=backend_name)
    pbar.update(1)
    

  8%|▊         | 1/12 [00:00<00:01,  9.15it/s, backend=PYT, dataset=adult-income]

125/125 [==============================] - 0s 196us/step


 67%|██████▋   | 8/12 [00:00<00:00, 22.20it/s, backend=sklearn, dataset=compas-recidivism]

Distribution of predictions: [501 499]
Exact match with y_test: 1.0
Sample mismatches:
Pred: 0, True: 0
Pred: 0, True: 0
Pred: 0, True: 0
Pred: 0, True: 0
Pred: 0, True: 0
Pred: 0, True: 0
Pred: 0, True: 0
Pred: 0, True: 0
Pred: 0, True: 0
Pred: 0, True: 0
Distribution of predictions: [481 500]
Exact match with y_test: 1.0
Sample mismatches:
Pred: 0, True: 0
Pred: 0, True: 0
Pred: 0, True: 0
Pred: 0, True: 0
Pred: 0, True: 0
Pred: 0, True: 0
Pred: 0, True: 0
Pred: 0, True: 0
Pred: 0, True: 0
Pred: 0, True: 0


100%|██████████| 12/12 [00:00<00:00, 22.20it/s, backend=PYT, dataset=compas-recidivism]   

...

125/125 [==============================] - 0s 182us/step


14it [00:00, 21.68it/s, backend=TF2, dataset=german-credit]

125/125 [==============================] - 0s 191us/step


100%|██████████| 12/12 [00:01<00:00, 11.54it/s, backend=sklearn, dataset=adult-income]

Distribution of predictions: [500 498]
Exact match with y_test: 1.0
Sample mismatches:
Pred: 0, True: 0
Pred: 0, True: 0
Pred: 0, True: 0
Pred: 0, True: 0
Pred: 0, True: 0
Pred: 0, True: 0
Pred: 0, True: 0
Pred: 0, True: 0
Pred: 0, True: 0
Pred: 0, True: 0
Distribution of predictions: [494 499]
Exact match with y_test: 1.0
Sample mismatches:
Pred: 0, True: 0
Pred: 0, True: 0
Pred: 0, True: 0
Pred: 0, True: 0
Pred: 0, True: 0
Pred: 0, True: 0
Pred: 0, True: 0
Pred: 0, True: 0
Pred: 0, True: 0
Pred: 0, True: 0


In [ ]:
rows = []
for (dataset, backend), info in cfe_performance_results.items():
    metrics = info['model_metrics']
    row = {
        'dataset': dataset,
        'backend': backend,
        'accuracy': round(metrics['accuracy'], 2),
        'f1_score': round(metrics['f1_score'], 2),
        'recall': round(metrics['recall'], 2),
        'precision': round(metrics['precision'], 2),
        'auc': round(metrics['auc'], 2)
    }
    rows.append(row)

df = pd.DataFrame(rows)

backend_order = {'sklearn': 0, 'PYT': 1, 'TF2': 2}
df['backend_priority'] = df['backend'].map(backend_order)
df_sorted = df.sort_values(by=['dataset', 'backend_priority'])
df_sorted = df_sorted.drop(columns=['backend_priority'])
df_sorted = df_sorted.reset_index()
df_sorted.drop(columns="index", inplace=True)
print(df_sorted)

              dataset  backend  accuracy  f1_score  recall  precision   auc
0        adult-income  sklearn      1.00      1.00    1.00       1.00  1.00
1        adult-income      PYT      0.47      0.42    0.47       0.45  0.46
2        adult-income      TF2      0.51      0.26    0.18       0.53  0.51
3   compas-recidivism  sklearn      1.00      1.00    1.00       1.00  1.00
4   compas-recidivism      PYT      0.87      0.87    0.87       0.88  0.94
5   compas-recidivism      TF2      0.38      0.54    0.56       0.52  0.52
6       german-credit  sklearn      1.00      1.00    1.00       1.00  1.00
7       german-credit      PYT      0.67      0.65    0.67       0.69  0.75
8       german-credit      TF2      0.53      0.34    0.26       0.48  0.49
9        lending-club  sklearn      1.00      1.00    1.00       1.00  1.00
10       lending-club      PYT      0.48      0.33    0.49       0.25  0.23
11       lending-club      TF2      0.70      0.58    0.69       0.49  0.49


In [ ]:
rows = []
for (dataset, backend), info in cfe_performance_results.items():
    metrics = info['model_metrics']
    row = {
        'dataset': dataset,
        'backend': backend,
        'accuracy': round(metrics['accuracy'], 2),
        'f1_score': round(metrics['f1_score'], 2),
        'recall': round(metrics['recall'], 2),
        'precision': round(metrics['precision'], 2),
        'auc': round(metrics['auc'], 2)
    }
    rows.append(row)

df = pd.DataFrame(rows)

backend_order = {'sklearn': 0, 'PYT': 1, 'TF2': 2}
df['backend_priority'] = df['backend'].map(backend_order)
df_sorted = df.sort_values(by=['dataset', 'backend_priority'])
df_sorted = df_sorted.drop(columns=['backend_priority'])
df_sorted = df_sorted.reset_index()
df_sorted.drop(columns="index", inplace=True)
print(df_sorted)

NameError: name 'cfe_performance_results' is not defined

In [ ]:
# Counterfactual generation time table
time_spent_data = {
    dataset: {
        model: {
            'time': round(details['time'], 2)
        }
        for model, details in models.items()
    }
    for dataset, models in benchmarking_results.items()
}

rows = []

for dataset, models in time_spent_data.items():
    for model_name, model_info in models.items():
        row = {
            'Dataset': dataset,
            'Model': 'Genetic' if model_name == 'sklearn' else f'Gradient ({model_name})',
            'Time (ms)': model_info.get('time'),
        }
        rows.append(row)

time_df = pd.DataFrame(rows)
df_sorted['Time (s)'] = time_df['Time (ms)']
markdown_table = df_sorted.to_markdown(index=False)
print(markdown_table)

In [ ]:
# metric functions
def compute_validity(exp: dice_ml_x.Dice) -> float:
    return exp.get_validity_percentage()

def compute_mad(data_class: dice_ml_x.Data, normalized=False) -> dict:
    return data_class.get_valid_mads(normalized=normalized)

def compute_continuous_proximity(C: pd.DataFrame, x: pd.DataFrame, data_class: dice_ml_x.Data) -> float:
    mads = compute_mad(data_class)
    total_proximity = 0.0

    for feature in data_class.continuous_feature_names:
        diff = np.abs(C[feature] - x[feature].iloc[0])
        total_proximity += diff / mads[feature]

    return -np.mean(total_proximity)

def compute_continuous_proximity(C: pd.DataFrame, x: pd.DataFrame, data_class: dice_ml_x.Data) -> float:
    mads = compute_mad(data_class)
    normalized_diff = (np.abs(C[data_class.continuous_feature_names] - x[data_class.continuous_feature_names].iloc[0]) / 
                      np.array([mads[feature] for feature in data_class.continuous_feature_names]).reshape(1, -1))
    mean_per_CF = np.nanmean(normalized_diff, axis=1)
    return -np.mean(mean_per_CF)

def compute_categorical_proximity(C: pd.DataFrame, x: pd.DataFrame, data_class: dice_ml_x.Data) -> float:
    categorical_feats = data_class.categorical_feature_names
    if len(x) == 0 or len(categorical_feats) == 0:
        return 0.0
    x_values: pd.Series = x.iloc[0]
    diff_matrix: pd.DataFrame = C[categorical_feats] != x_values[categorical_feats]
    diff_count: pd.Series = diff_matrix.sum(axis=1)
    d_cat = len(categorical_feats)

    average_distance = diff_count.mean() / (d_cat * len(C)) if d_cat > 0 else 0.0

    return 1 - average_distance

def compute_continuous_diversity(C: pd.DataFrame, data_class: dice_ml_x.Data) -> float:
    cont_feats = data_class.continuous_feature_names

    if isinstance(C, pd.DataFrame):
        X = C[cont_feats].values
    else:
        X = C

    k, d = X.shape

    if k < 2 or d == 0:
        return 0.0
    
    mad_dict = compute_mad(data_class)
    mad_vector = np.array([
        mad_dict[feature] if mad_dict[feature] != 0 else 1.0
        for feature in cont_feats
    ]).reshape(1, d)
    
    diff = np.abs(X[:, None, :] - X[None, :, :])
    normalized_diff = diff / mad_vector

    pairwise_dist = np.mean(normalized_diff, axis=2)
    triu_indices = np.triu_indices(k, k=1)
    if len(triu_indices[0]) == 0:
        return 0.0
    average_distance = np.mean(pairwise_dist[triu_indices])
    return average_distance

def compute_continuous_diversity(C: pd.DataFrame, data_class: dice_ml_x.Data) -> float:
    mads = compute_mad(data_class)  
    X = C[data_class.continuous_feature_names].values
    diff = np.abs(X[:, None, :] - X[None, :, :])
    normalized_diff = diff / np.array([mads[feature] for feature in data_class.continuous_feature_names]).reshape(1, 1, -1)
    pairwise_dist = np.nanmean(normalized_diff, axis=2)
    triu_indices = np.triu_indices(len(C), k=1)
    return np.mean(pairwise_dist[triu_indices]) if len(triu_indices[0]) > 0 else 0.0

def compute_categorical_diversity(C: pd.DataFrame, data_class: dice_ml_x.Data) -> float:
    cat_feats = data_class.categorical_feature_names

    if isinstance(C, pd.DataFrame):
        X = C[cat_feats].values
    else:
        X = C
    
    k, d = X.shape

    if k < 2 or d == 0:
        return 0.0

    diff = (X[:, None, :] != X[None, :, :]).astype(np.float32)

    pairwise_dist = np.mean(diff, axis=2)
    triu_indices = np.triu_indices(k, k=1)
    if len(triu_indices[0]) == 0:
        return 0.0
    average_distance = np.mean(pairwise_dist[triu_indices])
    return average_distance

def compute_count_diversity(C: pd.DataFrame):
    if isinstance(C, pd.DataFrame):
        X = C.values
    else:
        X = C

    k, d = X.shape

    if X.size == 0 or k < 2 or d == 0:
        return 0.0
    
    diff = (X[:, None, :] != X[None, :, :]).astype(np.float32)
    triu_indices = np.triu_indices(k, k=1)
    total_diff = np.sum(diff[triu_indices[0], triu_indices[1], :])

    n_pairs = len(triu_indices[0])
    return total_diff / (n_pairs * d)

def compute_sparsity(C: pd.DataFrame, x: pd.DataFrame, data_class: dice_ml_x.Data) -> float:
    cont_feats = data_class.continuous_feature_names
    cont_CFs = C[cont_feats].to_numpy()
    cont_X = x[cont_feats].to_numpy()

    k, d = cont_CFs.shape

    diff = (cont_CFs != cont_X[0])

    num_changed = diff.sum()

    return 1 - (num_changed / (k * d))

def do_perturbation(x_ohe: pd.DataFrame, data_class: dice_ml_x.Data):
    cat_cols = data_class.get_encoded_categorical_feature_indexes()
    cat_cols = [col for group in cat_cols for col in group]
    x_tensor = torch.tensor(x_ohe.values, dtype=torch.float32)
    
    continuous_feature_indexes = list(set(list(range(len(x_ohe.columns)))) - set(cat_cols))
    categorical_feature_indexes = data_class.get_encoded_categorical_feature_indexes()
    if continuous_feature_indexes:
        
        continuous_slice = x_tensor[:, continuous_feature_indexes]
        noise = continuous_slice * 0.1
        noise_mask = torch.zeros_like(x_tensor)
        noise_mask[:, continuous_feature_indexes] = noise
        x_tensor = x_tensor + noise_mask

    if categorical_feature_indexes:
        for cat_cols in categorical_feature_indexes:
            cat_slice = x_tensor[:, cat_cols]
            sample_size = cat_slice.shape[0]
            num_cats = cat_slice.shape[1]

            rand_idx = torch.randint(low=0, high=num_cats, size=(sample_size, ))

            cat_slice_perturbed = torch.nn.functional.one_hot(rand_idx, num_classes=num_cats).float()

            cat_mask = torch.zeros_like(x_tensor)
            cat_mask[:, cat_cols] = cat_slice_perturbed
            x_perturbed = x_tensor + cat_mask
    return x_perturbed

def generate_perturbations(x_ohe: pd.DataFrame, model: any, data_class: dice_ml_x.Data,
                           max_iter=100, tol=1e-3, gamma=1e-2):
    x_ohe_tensor = torch.tensor(x_ohe.values, dtype=torch.float32, requires_grad=True)
    x_perturbed = do_perturbation(x_ohe, data_class)
    perturbation_optimizer = torch.optim.Adam([x_perturbed], lr=1e-3)

    prev_loss = np.inf
    for _ in range(max_iter):
        with torch.no_grad():
            model.model.eval()
            pred_i = model.model(x_ohe_tensor)
            pred_i_prime = model.model(x_perturbed)
        class_loss = torch.mean((pred_i - pred_i_prime) ** 2)
        distance = torch.norm(x_perturbed - x_ohe_tensor, p=2)
        loss = class_loss + gamma * distance

        perturbation_optimizer.zero_grad()
        loss.backward()

        perturbation_optimizer.step()
        if abs(loss.item() - prev_loss) < tol:
            break
        prev_loss = loss.item()
    return x_perturbed.detach()

def compute_robustness(C: pd.DataFrame, x: pd.DataFrame, data_class: dice_ml_x.Data,
                       explainer: dice_ml_x.Dice, exp_options: OrderedDict, model: any,
                       target_name: str) -> float:
    x_ohe = data_class.get_ohe_min_max_normalized_data(x)
    x_prime_ohe_tensor = generate_perturbations(x_ohe, model, data_class)
    x_prime_decoded = data_class.get_decoded_data(x_prime_ohe_tensor.numpy())
    x_prime = data_class.get_inverse_ohe_min_max_normalized_data(x_prime_decoded)
    C_prime = explainer.generate_counterfactuals(x_prime, **exp_options)
    C_prime = C_prime.to_dataframe()
    na_cols = C_prime.columns[C_prime.isna().any().tolist()].tolist()
    C_prime[na_cols] = C_prime[na_cols].fillna(x_prime[na_cols].iloc[0])
    C_prime[target_name] = (C_prime[target_name] >= 0.5).astype(int)
    C_ohe_tensor = torch.tensor(d.get_ohe_min_max_normalized_data(C).values, dtype=torch.float32)
    C_prime_ohe_tensor = torch.tensor(d.get_ohe_min_max_normalized_data(C_prime).values, dtype=torch.float32)
    return torch.mean(torch.cdist(C_ohe_tensor, C_prime_ohe_tensor, p=2)).item()


In [ ]:
METRIC_KEYS = [
    'validity',
    'cat_diversity',
    'cont_diversity',
    'cont_count_diversity',
    'cat_proximity',
    'cont_proximity',
    'sparsity',
    'robustness',
    'num_cfs',
    'iteration'
]

def empty_metrics_dict():
    return {key: [] for key in METRIC_KEYS}

def compute_all_metrics(cf_df, query_instance_with_target, query_instance, d, model, exp, exp_options, target_name, it):
    cf_df[target_name] = np.where(cf_df[target_name] < 0.5, 0, 1)
    metrics = {}
    metrics['validity'] = compute_validity(cf_df, target_name)
    metrics['cat_diversity'] = compute_categorical_diversity(cf_df, d)
    metrics['cont_diversity'] = compute_continuous_diversity(cf_df, d)
    metrics['cont_count_diversity'] = compute_count_diversity(cf_df)
    metrics['cat_proximity'] = compute_categorical_proximity(cf_df, query_instance_with_target, d)
    metrics['cont_proximity'] = compute_continuous_proximity(cf_df, query_instance_with_target, d)
    metrics['sparsity'] = compute_sparsity(cf_df, query_instance_with_target, d)
    metrics['robustness'] = compute_robustness(C=cf_df, x=query_instance,
                                               data_class=d, explainer=exp, exp_options=exp_options,
                                               model=model, target_name=target_name)
    metrics['num_cfs'] = len(cf_df)
    metrics['iteration'] = it
    metrics['cf_df'] = cf_df
    return metrics

datasets = [(helpers.load_compas_dataset(), "twoyearrecid", "compas-recidivism"),
            (helpers.load_adult_income_dataset(), "income", "adult-income"),
            (helpers.load_lending_club_dataset(), "loan_status", "lending-club"),
            (helpers.load_german_credit_dataset(), "credit_risk", "german-credit")]

backends = ['PYT']
comparison_results = {}

for df, target_name, df_name in tqdm(datasets, desc="Datasets"):
    comparison_results[df_name] = {}
    df_target = df[target_name]
    train_dataset, test_dataset, y_train, y_test = train_test_split(df, df_target, test_size=0.2,
                                                                    random_state=42, stratify=df_target)
    n = 500
    if df_name == "german-credit":
        n = 200

    test_dataset_samples = test_dataset.sample(n=n)

    x_test = test_dataset_samples.drop(columns=[target_name]).copy()

    cont_feats = df.select_dtypes(include=[np.number]).columns.difference([target_name]).tolist()

    d = dice_ml_x.Data(dataframe=train_dataset, continuous_features=cont_feats, outcome_name=target_name)

    for backend_name in tqdm(backends, desc=f"{df_name} Backends", leave=False):
        comparison_results[df_name][backend_name] = {}

        model_options = OrderedDict(model=models[df_name][backend_name], backend=backend_name)
        method = 'genetic' if backend_name == 'sklearn' else 'gradient'
        if backend_name != 'sklearn':
            model_options['func'] = 'ohe-min-max'
            if backend_name == 'PYT':
                model_options['model'] = models[df_name][backend_name].model

        m = dice_ml_x.Model(**model_options)
        exp1 = dice_ml_x.Dice(d, m, method=method)

        algo_variants = {
            'diverse_cfs': {"algorithm": "DiverseCF", "diversity_weight": 1.0,
                            "proximity_weight": 0.5, "posthoc_sparsity_param": 0.0},
            'nodiverse_cfs': {"algorithm": "DiverseCF", "diversity_weight": 0.0,
                              "proximity_weight": 0.5, "posthoc_sparsity_param": 0.0},
            'randominit_cfs': {"algorithm": "RandomInitCF", "diversity_weight": 1.0,
                               "proximity_weight": 0.5, "posthoc_sparsity_param": 0.0},
            'diverse_cfs_sparse': {"algorithm": "DiverseCF", "diversity_weight": 1.0,
                                   "proximity_weight": 0.5, "posthoc_sparsity_param": 0.1},
            'nodiverse_cfs_sparse': {"algorithm": "DiverseCF", "diversity_weight": 0.0,
                                     "proximity_weight": 0.5, "posthoc_sparsity_param": 0.1},
            'randominit_cfs_sparse': {"algorithm": "RandomInitCF", "diversity_weight": 1.0,
                                      "proximity_weight": 0.5, "posthoc_sparsity_param": 0.1},
        }

        for variant in list(algo_variants.keys()) + ['single_cf']:
            comparison_results[df_name][backend_name][variant] = empty_metrics_dict()

        pbar = tqdm(algo_variants.items(), desc='Variants', leave=True)

        single_cf_found = False

        while single_cf_found == False:
            rand_idx = random.randrange(0, len(x_test))
            query_instance = x_test.iloc[[rand_idx]]
            query_instance_with_target = test_dataset_samples.iloc[[rand_idx]]

            
            exp_options = OrderedDict(
                total_CFs=1,
                desired_class='opposite',
                posthoc_sparsity_param=0.0,
                proximity_weight=0.5,
                diversity_weight=1.0,
                robustness_weight=5.0,
                algorithm='DiverseCF'
            )
            try:
                cf_obj = exp1.generate_counterfactuals(query_instance, **exp_options)

                cf_df = cf_obj.to_dataframe()
                metrics = compute_all_metrics(cf_df=cf_df, query_instance_with_target=query_instance_with_target,
                                            query_instance=query_instance, d=d, model=m,
                                            exp=exp1, exp_options=exp_options,
                                            target_name=target_name, it=1)
                for key in METRIC_KEYS:
                    comparison_results[df_name][backend_name]['single_cf'][key].append(metrics[key])
                single_cf_found = True
            except Exception as e:
                continue

        for variant, params in pbar:
            it = 1
            while it <= 10:
                exp1 = dice_ml_x.Dice(d, m, method=method)
                rand_idx = random.randrange(0, len(x_test))
                query_instance = x_test.iloc[[rand_idx]]
                query_instance_with_target = test_dataset_samples.iloc[[rand_idx]] 
                total_CFs = it

                diversity_weight = float(params.get('diversity_weight', 1.0))
                proximity_weight = float(params.get('proximity_weight', 0.5))
                posthoc_sparsity_param = float(params.get('posthoc_sparsity_param', 0.0))
                algorithm = params.get('algorithm')
                
                exp_options = OrderedDict(
                    total_CFs=total_CFs,
                    algorithm=algorithm,
                    desired_class='opposite',
                    diversity_weight=diversity_weight,
                    proximity_weight=proximity_weight,
                    robustness_weight=5.0,
                    posthoc_sparsity_param=posthoc_sparsity_param
                )
                try:
                    cf_obj = exp1.generate_counterfactuals(
                        query_instances=query_instance,
                        **exp_options
                    )

                    cf_df = cf_obj.to_dataframe()
                    metrics = compute_all_metrics(cf_df=cf_df, query_instance_with_target=query_instance_with_target,
                                                query_instance=query_instance, d=d, model=m,
                                                exp=exp1, exp_options=exp_options,
                                                target_name=target_name, it=it)
                    for key in METRIC_KEYS:
                        comparison_results[df_name][backend_name][variant][key].append(metrics[key])
                    it += 1
                except Exception as e:
                    continue

        pbar.set_postfix(OrderedDict(backend=backend_name))
        pbar.update(1)

In [ ]:
# If you run the cell above uncomment the two lines below to save results into a pickle file
import pickle

# DANGER! Make sure you already run the cell above before uncommenting the following lines.
#with open("comparison_figure_1.pkl", "wb") as comp_file:
#    pickle.dump(comparison_results, comp_file)

with open("comparison_figure_1.pkl", "rb") as comp_file:
    comparison_results = pickle.load(comp_file)

In [ ]:
METRIC_KEYS = [
    'validity',
    'cat_diversity',
    'cont_diversity',
    'cont_count_diversity',
    'cat_proximity',
    'cont_proximity',
    'sparsity',
    'robustness',
    'num_cfs'
]

rows = []
for dataset, dataset_data in comparison_results.items():
    
    if 'PYT' not in dataset_data:
        continue
    pyt_data = dataset_data['PYT']

    for variant, metrics in pyt_data.items():
    
        iterations = metrics.get("iteration", list(range(1, len(metrics["validity"]) + 1)))
    
        for i, iter_val in enumerate(iterations):
            row = {
                "dataset": dataset,
                "backend": "PYT",
                "variant": variant,
                "iteration": iter_val
            }
    
            for key in METRIC_KEYS:
    
                row[key] = metrics[key][i] if i < len(metrics[key]) else None
            rows.append(row)

df_table = pd.DataFrame(rows)


df_table = df_table.sort_values(by=["dataset", "variant", "iteration"])
df_table[df_table['dataset'] == 'compas-recidivism']

In [ ]:
import math

def normalize_list(values, target_min, target_max):
    obs_min = min(values)
    obs_max = max(values)
    if obs_max == obs_min:
        return [ (target_min + target_max) / 2.0 for _ in values ]
    scale = (target_max - target_min) / (obs_max - obs_min)
    return [ (v - obs_min) * scale + target_min for v in values ]

def add_margin(target_range, margin=0.05):
    tmin, tmax = target_range
    delta = tmax - tmin
    return (tmin - margin * delta, tmax + margin * delta)

target_ranges = {
    "validity": (0, 100),
    "cat_diversity": (0, 1),
    "cont_diversity": (0, 6),
    "cont_count_diversity": (0, 1),
    "cat_proximity": (0, 1),
    "cont_proximity": (-5, 0),
    "sparsity": (0, 1),
    "robustness": (0, 5)
}

line_styles = {
    "diverse_cfs": ('-', 'D', 'blue'),
    "diverse_cfs_sparse": ('--', 'D', 'blue'),
    "nodiverse_cfs": ('-', 'o', 'orange'),
    "nodiverse_cfs_sparse": ('--', 'o', 'orange'),
    "randominit_cfs": ('-', 'x', 'green'),
    "randominit_cfs_sparse": ('--', 'x', 'green'),
    "single_cf": ('-', 's', 'red')
}

metric_names = ['% Valid CFs', 'Categorical-Diversity', 'Continuous-Diversity',
                'Cont-Count-Diversity', 'Categorical-Proximity', 'Continuous-Proximity', 'Continuous-Sparsity', 'Robustness']

datasets_list = ['adult-income', 'lending-club', 'german-credit', 'compas-recidivism']

variants = list(comparison_results[datasets_list[2]]['PYT'].keys())


fig, axes = plt.subplots(nrows=len(datasets_list), ncols=len(target_ranges), 
                           figsize=(25, 16), sharex=False)
plt.subplots_adjust(hspace=0.4, wspace=0.3)

iterations_to_show = [1, 2, 4, 6, 8, 10]

for row_idx, dataset in enumerate(datasets_list):
    if dataset not in comparison_results:
        continue
    if 'PYT' not in comparison_results[dataset]:
        continue
    pyt_data = comparison_results[dataset]['PYT']
    
    for col_idx, key in enumerate(target_ranges.keys()):
        ax = axes[row_idx, col_idx]
        for variant in variants:
            variant_data = pyt_data[variant]
            if key not in variant_data:
                continue

            y_values = variant_data[key]
            x_values = []
            y_plot =[]

            for i in iterations_to_show:
                if i <= len(y_values):
                    x_values.append(i) 
                    y_plot.append(y_values[i - 1])

            if not x_values:
                continue

            if key == "validity":
                
                norm_values = [v * 100 for v in y_plot]
                
            else:
                norm_values = y_plot

            linestyle, marker, marker_color = line_styles.get(variant, ('-', 'o', 'blue'))    
            ax.plot(x_values, norm_values, markerfacecolor=marker_color, linestyle=linestyle,
                    marker=marker, label=variant)
        
        ax.set_ylim(add_margin(target_ranges[key], margin=0.05))
        
        ax.set_xticks(iterations_to_show)
        if row_idx == len(datasets_list) - 1:
            ax.set_xticklabels(iterations_to_show, fontsize=10)
        else:
            ax.set_xticklabels([])

        if col_idx == 0:
            ax.set_ylabel(dataset, fontsize=12, fontweight='bold')
        else:
            ax.set_ylabel("")
    
        if row_idx == 0:
            ax.set_title(metric_names[col_idx], fontsize=14, fontweight='bold')
        
        ax.grid(True)

n_variants = len(variants)
ncol = math.ceil(n_variants / 2)
handles, labels = axes[-1, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=ncol, bbox_to_anchor=(0.1, 0.0, 0.8, 0.1), mode="expand", fontsize=16)

plt.tight_layout(rect=[0, 0.05, 1, 0.95])
root_folder = 'figure_artefacts'
if not os.path.exists(root_folder):
    os.makedirs(root_folder)
fig_file_path = os.path.join(root_folder, 'comparison_fig.eps')
plt.savefig(fig_file_path, format='eps')
plt.show()

### Results on counterfactual datasets

The counterfactual datasets generated with 4 datasets and 3 different counterfactual generation strategies is investigated in terms of validity, diversity, and robustness.

In [ ]:
import os
file_names = os.listdir('cfe_datasets')
keys = [(f_name.split('_')[0], f_name.split('_')[1]) for f_name in file_names]
target_names = {'lending-club': 'loan_status', 'compas-recidivism': 'twoyearrecid',
                'adult-income': 'income', 'german-credit': 'credit_risk'}
cfe_datasets = {}

for key, f_name in zip(keys, file_names):
    new_key = key + (target_names[key[0]],)
    cfe_datasets[new_key] = pd.read_csv(os.path.join('cfe_datasets', f_name))

# Convert target column to binary (0 or 1 depending on whether its value is greater or less than 0.5)
for (df_name, backend_name, target_name), cfe_dataset in cfe_datasets.items():
    cfe_dataset[target_name] = np.where(cfe_dataset[target_name] < 0.5, 0, 1)

In [ ]:
def load_torch_model(model_path):
    print('model_path is -> ', model_path)
    dummy_state_dict = torch.load(model_path)
    dummy_state_dict = {f'model.{key}': value for key, value in dummy_state_dict.items()}
    in_features = dummy_state_dict['model.0.weight'].shape[1]
    model = neuralnetworks.PYTModel(in_features)
    model.load_state_dict(dummy_state_dict)
    return model

def load_tensorflow_model(model_path):
    model = neuralnetworks.TF2Model()
    model.load_weights(model_path)
    return model
root_dir = 'cfe_models'
file_names = [f_name for f_name in os.listdir(root_dir) if f_name.endswith(('pkl', 'pth', 'index')) and f_name != 'model_metrics_result.pkl']
keys = [(f_name.split('_')[0], f_name.split('_')[1]) for f_name in file_names if f_name.endswith(('pkl', 'pth', 'index')) and f_name != 'model_metrics_result.pkl']

dataset_names = [
    "compas-recidivism",
    "adult-income",
    "lending-club",
    "german-credit"
]

cfe_models = {'adult-income': {}, 'lending-club': {},
          'german-credit': {}, 'compas-recidivism': {}}

def pick_model(models, df_name, backend_name, root_dir, f_name):
    if backend_name == 'sklearn':
        model_path = os.path.join(root_dir, f_name)
        with open(model_path, 'rb') as sklearn_model_file:
            sklearn_model = pickle.load(sklearn_model_file)
        models[df_name][backend_name] = sklearn_model
    elif backend_name == 'PYT':
        model_path = os.path.join(root_dir, f_name)
        models[df_name][backend_name] = load_torch_model(model_path)
    elif backend_name == 'TF2':
        model_path = os.path.join(root_dir, f_name)
        models[df_name][backend_name] = load_tensorflow_model(model_path)


for (df_name, backend_name), f_name in zip(keys, file_names):
    print((df_name, backend_name), ' -> ', f_name)
    if df_name == 'adult-income':
        pick_model(cfe_models, df_name, backend_name, root_dir, f_name)
    elif df_name == 'lending-club':
        pick_model(cfe_models, df_name, backend_name, root_dir, f_name)
    elif df_name == 'german-credit':
        pick_model(cfe_models, df_name, backend_name, root_dir, f_name)
    elif df_name == 'compas-recidivism':
        pick_model(cfe_models, df_name, backend_name, root_dir, f_name)

In [ ]:
cfe_dfs_validity_results = {}
cfe_dfs_cont_diversity_results = {}
cfe_dfs_cat_diversity_results = {}
for (df_name, backend, target_name), cfe_dataset in cfe_datasets.items():
    cfe_dfs_validity_results[(df_name, backend)] = compute_validity(cfe_dataset, target_name)
    target_col = cfe_dataset[target_name]
    train_dataset, test_dataset, y_train, y_test = train_test_split(cfe_dataset, target_col, test_size=0.2,
                                                                    random_state=42, shuffle=True, stratify=target_col)
    cont_feats = cfe_dataset.select_dtypes(include=[np.number]).columns.difference([target_name]).tolist()
    data_class = dice_ml_x.Data(dataframe=train_dataset, continuous_features=cont_feats, outcome_name=target_name)
    cfe_dfs_cont_diversity_results[(df_name, backend)] = compute_continuous_diversity(cfe_dataset, data_class)
    cfe_dfs_cat_diversity_results[(df_name, backend)] = compute_categorical_diversity(cfe_dataset, data_class)
print(cfe_dfs_validity_results)
print(cfe_dfs_cont_diversity_results)
print(cfe_dfs_cat_diversity_results)


In [ ]:
dummy_dict = {'adult-income': {}, 'lending-club': {},
               'german-credit': {}, 'compas-recidivism': {}}
for (df_name, backend_name), cont_result in cfe_dfs_cont_diversity_results.items():
    if df_name == 'adult-income':
        dummy_dict['adult-income'][backend_name] = round(cont_result, 2)
    elif df_name == 'lending-club':
        dummy_dict['lending-club'][backend_name] = round(cont_result, 2)
    elif df_name == 'german-credit':
        dummy_dict['german-credit'][backend_name] = round(cont_result, 2)
    elif df_name == 'compas-recidivism':
        dummy_dict['compas-recidivism'][backend_name] = round(cont_result, 2)
        
table_mkdwn = '| Dataset            | sklearn | PYT  | TF2  |\n'
table_mkdwn += '|--------------------|---------|------|------|\n'

for dataset, models in dummy_dict.items():
    table_mkdwn += f'| {dataset:<18} | {models["sklearn"]:.2f} | {models["PYT"]:.2f} | {models["TF2"]:.2f} |\n'

print(table_mkdwn)

In [ ]:
dummy_dict = {'adult-income': {}, 'lending-club': {},
               'german-credit': {}, 'compas-recidivism': {}}
for (df_name, backend_name), cont_result in cfe_dfs_cat_diversity_results.items():
    if df_name == 'adult-income':
        dummy_dict['adult-income'][backend_name] = round(cont_result, 2)
    elif df_name == 'lending-club':
        dummy_dict['lending-club'][backend_name] = round(cont_result, 2)
    elif df_name == 'german-credit':
        dummy_dict['german-credit'][backend_name] = round(cont_result, 2)
    elif df_name == 'compas-recidivism':
        dummy_dict['compas-recidivism'][backend_name] = round(cont_result, 2)
        
table_mkdwn = '| Dataset            | sklearn | PYT  | TF2  |\n'
table_mkdwn += '|--------------------|---------|------|------|\n'

for dataset, models in dummy_dict.items():
    table_mkdwn += f'| {dataset:<18} | {models["sklearn"]:.2f} | {models["PYT"]:.2f} | {models["TF2"]:.2f} |\n'

print(table_mkdwn)

In [ ]:
def compute_robustness_from_perturbation(C: pd.DataFrame, target_name: str, data_class: dice_ml_x.Data,
                                         backend_name: str, model: any, results: dict, noise_factor=0.05):
    """Estimate robustness by perturbing the counterfactuals and checking stability."""
    C_targetless = C.drop(columns=[target_name])
    y_truth = C[target_name].values
    C_ohe_tensor_targetless = data_class.get_ohe_min_max_normalized_data(C_targetless).values
    
    perturbed_cf = C_ohe_tensor_targetless + np.random.normal(0, noise_factor, C_ohe_tensor_targetless.shape)
    
    if backend == 'PYT':
        perturbed_cf = torch.tensor(perturbed_cf, dtype=torch.float32)
        
        pred = model(perturbed_cf)
        y_truth_tensor = y_truth.reshape(y_truth.shape[0], 1)
        pred = pred.detach().numpy()
        pred = np.where(pred < 0.5, 0, 1)
        results[df_name][backend_name] = (y_truth_tensor == pred).astype(int).sum() / len(pred)
    elif backend == 'TF2':
        pred = model.predict(perturbed_cf)
        y_truth_tensor = y_truth.reshape(y_truth.shape[0], 1)
        pred = np.where(pred < 0.5, 0, 1)
        results[df_name][backend_name] = (y_truth_tensor == pred).astype(int).sum() / len(pred)
    else:
        data_class.create_ohe_params(data_class.get_ohe_min_max_normalized_data(C_targetless))
        C_ohe_decoded_targetless = data_class.get_decoded_data(perturbed_cf)
        C_targetless_noised = data_class.get_inverse_ohe_min_max_normalized_data(C_ohe_decoded_targetless)
        pred = model.predict(C_targetless_noised)
        results[df_name][backend_name] = (y_truth == pred).astype(int).sum() / len(pred)

cfe_robustness_results = {'adult-income': {}, 'lending-club': {},
                          'german-credit': {}, 'compas-recidivism': {}}

for (df_name, backend, target_name), cfe_dataset in cfe_datasets.items():
    target_col = cfe_dataset[target_name]
    train_dataset, test_dataset, y_train, y_test = train_test_split(cfe_dataset, target_col, test_size=0.2,
                                                                    random_state=42, shuffle=True, stratify=target_col)
    cont_feats = cfe_dataset.select_dtypes(include=[np.number]).columns.difference([target_name]).tolist()
    data_class = dice_ml_x.Data(dataframe=train_dataset, continuous_features=cont_feats, outcome_name=target_name)
    
    compute_robustness_from_perturbation(cfe_dataset, target_name, data_class, backend, cfe_models[df_name][backend], cfe_robustness_results)

In [ ]:
table_mkdwn = '| Robustness Score     | sklearn | PYT  | TF2  |\n'
table_mkdwn += '|--------------------|---------|------|------|\n'

for dataset_name, rob_dict in cfe_robustness_results.items():
    table_mkdwn += f'| {dataset_name:<18} | {rob_dict["sklearn"]:.2f} | {rob_dict["PYT"]:.2f} | {rob_dict["TF2"]:.2f} |\n'

print(table_mkdwn)

The validity of each dataset for different backend algorithms are as below:

| VALIDITY SCORE    | sklearn | PYT  | TF2  |
|-------------------|---------|------|------|
| adult-income      | 0.89    | 0.97 | 0.99 |
| lending-club      | 0.77    | 0.76 | 0.87 |
| german-credit     | 0.45    | 0.48 | 0.99 |
| compas-recidivism | 0.66    | 0.58 | 0.90 |

The continuous diversity of each dataset for different backend algorithms are as below:

| Continuous Diversity | sklearn | PYT  | TF2  |
|----------------------|---------|------|------|
| adult_income         | 2.35    | 1.62 | 1.68 |
| lending-club         | 1.83    | 2.31 | 5.39 |
| german-credit        | 1.21    | 1.42 | 1.37 |
| compas-recidivism    | 2.97    | 2.51 | 1.89 |

The categorical diversity of each dataset for different backend algorithms are as below:

| Categorical Diversity | sklearn | PYT  | TF2  |
|-----------------------|---------|------|------|
| adult_income          | 0.50    | 0.64 | 0.66 |
| lending-club          | 0.75    | 0.62 | 0.60 |
| german-credit         | 0.51    | 0.58 | 0.58 |
| compas-recidivism     | 0.43    | 0.49 | 0.48 |

To ensure the consistency for the desired classes and randomization we generated 5 counterfactuals until we reached the desired number for each classes. Unfortunately, because of lacking the original instances used for counterfactual generation it wasn't possible to compute an overall proximity score. However, although it's not possible to compute an aggregated score

| Robustness Score   | sklearn | PYT  | TF2  |
|--------------------|---------|------|------|
| adult-income       | 1.00    | 0.39 | 0.53 |
| lending-club       | 0.98    | 0.49 | 0.40 |
| german-credit      | 0.99    | 0.61 | 0.53 |
| compas-recidivism  | 1.00    | 0.29 | 0.27 |

### Visually comparing the original dataset and counterfactual dataset

Here we will show how the original datasets and counterfactual datasets lie on the plot to analyze whether the counterfactuals are within the original datasets boundaries. We first plot the PCA chart.

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

dataset_to_col = {
    "compas-recidivism": 0,
    "adult-income": 1,
    "lending-club": 2,
    "german-credit": 3
}

backend_to_row = {
    "sklearn": 0,
    "PYT": 1,
    "TF2": 2
}


all_datasets = {
    "compas-recidivism": {
        "data": helpers.load_compas_dataset(),
        "target": "twoyearrecid"
    },
    "adult-income": {
        "data": helpers.load_adult_income_dataset(),
        "target": "income"
    },
    "lending-club": {
        "data": helpers.load_lending_club_dataset(),
        "target": "loan_status"
    },
    "german-credit": {
        "data": helpers.load_german_credit_dataset(),
        "target": "credit_risk"
    }
}

root_dir = 'cfe_datasets'
cfe_files_list = os.listdir(root_dir)

fig, axes = plt.subplots(nrows=3, ncols=4, figsize=(16, 12))


all_handles, all_labels = None, None

for file_name in cfe_files_list:
    
    parts = file_name.split('_')
    dataset_name = parts[0]
    backend_name = parts[1].replace('.csv', '')

    row = backend_to_row[backend_name]
    col = dataset_to_col[dataset_name]
    ax = axes[row, col]

    original_data = all_datasets[dataset_name]["data"]
    target_name = all_datasets[dataset_name]["target"]

    n = 500
    if dataset_name == 'german-credit':
        n = 300
    _0_original_samples = original_data[original_data[target_name] < 0.5].sample(n=n)
    _1_original_samples = original_data[original_data[target_name] >= 0.5].sample(n=n)
    original_sample = pd.concat([_0_original_samples, _1_original_samples], ignore_index=True)

    orig_cont_feats = (
        original_sample
        .select_dtypes(include=[np.number])
        .columns
        .difference([target_name])
        .tolist()
    )

    orig_data_class = dice_ml_x.Data(
        dataframe=original_sample,
        continuous_features=orig_cont_feats,
        outcome_name=target_name
    )

    cfe_data = pd.read_csv(os.path.join(root_dir, file_name))
    combined_data = pd.concat([original_sample, cfe_data], ignore_index=True)
    X_combined = combined_data.drop(columns=[target_name])
    y_combined = combined_data[target_name]

    combined_data_class = dice_ml_x.Data(
        dataframe=combined_data,
        continuous_features=orig_cont_feats,
        outcome_name=target_name
    )

    combined_data_ohe = combined_data_class.get_ohe_min_max_normalized_data(X_combined)
    orig_len = len(original_sample)
    orig_sample_ohe = combined_data_ohe.iloc[:orig_len]
    cfe_data_ohe = combined_data_ohe.iloc[orig_len:]
    y_orig = y_combined[:orig_len]
    y_cfe = y_combined[orig_len:]

    pca = PCA(n_components=2)
    pca.fit(orig_sample_ohe)

    original_sample_2d = pca.transform(orig_sample_ohe)
    cfe_data_2d = pca.transform(cfe_data_ohe)

    cfe_cont_feats = (
        cfe_data
        .select_dtypes(include=[np.number])
        .columns
        .difference([target_name])
        .tolist()
    )
    
    y_orig = np.where(y_orig < 0.5, 0, 1)
    y_cfe = np.where(y_cfe < 0.5, 0, 1)

    unique_classes = np.unique(y_orig)

    for class_val in unique_classes:
        mask = (y_orig == class_val)
        ax.scatter(
            original_sample_2d[mask, 0],
            original_sample_2d[mask, 1],
            alpha=0.5,
            label=f'Original Data - Class {class_val}'
        )

    unique_classes_cfe = np.unique(y_cfe)

    for class_val in unique_classes_cfe:
        mask = (y_cfe == class_val)
        ax.scatter(
        cfe_data_2d[mask, 0],
        cfe_data_2d[mask, 1],
        alpha=0.5,
        marker='x',
        label=f'CF Data - Class {class_val}'
    )

    ax.set_title(f"{dataset_name} - {backend_name}", fontsize=10)
    ax.set_xlabel('PCA Component 1')
    ax.set_ylabel('PCA Component 2')

    if all_handles is None and all_labels is None:
        all_handles, all_labels = ax.get_legend_handles_labels()

if all_handles is not None and all_labels is not None:
    fig.legend(
        all_handles, all_labels,
        loc='lower center',
        ncol=2,
        bbox_to_anchor=(0.5, 0.01)
    )

plt.tight_layout(rect=[0, 0.05, 1, 1])
root_folder = 'figure_artefacts'
if not os.path.exists(root_folder):
    os.makedirs(root_folder)
fig_file_path = os.path.join(root_folder, 'pca_dice_x.eps')
plt.savefig(fig_file_path, format='eps')
plt.show()


To improve the visualization we plot the same graph also with t-SNE method.

In [ ]:
from sklearn.manifold import TSNE

dataset_to_col = {
    "compas-recidivism": 0,
    "adult-income": 1,
    "lending-club": 2,
    "german-credit": 3
}

backend_to_row = {
    "sklearn": 0,
    "PYT": 1,
    "TF2": 2
}


all_datasets = {
    "compas-recidivism": {
        "data": helpers.load_compas_dataset(),
        "target": "twoyearrecid"
    },
    "adult-income": {
        "data": helpers.load_adult_income_dataset(),
        "target": "income"
    },
    "lending-club": {
        "data": helpers.load_lending_club_dataset(),
        "target": "loan_status"
    },
    "german-credit": {
        "data": helpers.load_german_credit_dataset(),
        "target": "credit_risk"
    }
}

root_dir = 'cfe_datasets'
cfe_files_list = os.listdir(root_dir)

fig, axes = plt.subplots(nrows=3, ncols=4, figsize=(16, 12))


all_handles, all_labels = None, None

for file_name in cfe_files_list:
    
    parts = file_name.split('_')
    dataset_name = parts[0]
    backend_name = parts[1].replace('.csv', '')

    row = backend_to_row[backend_name]
    col = dataset_to_col[dataset_name]
    ax = axes[row, col]

    original_data = all_datasets[dataset_name]["data"]
    target_name = all_datasets[dataset_name]["target"]

    n = 500
    if dataset_name == 'german-credit':
        n = 300
    _0_original_samples = original_data[original_data[target_name] < 0.5].sample(n=n)
    _1_original_samples = original_data[original_data[target_name] >= 0.5].sample(n=n)
    original_sample = pd.concat([_0_original_samples, _1_original_samples], ignore_index=True)

    orig_cont_feats = (
        original_sample
        .select_dtypes(include=[np.number])
        .columns
        .difference([target_name])
        .tolist()
    )

    orig_data_class = dice_ml_x.Data(
        dataframe=original_sample,
        continuous_features=orig_cont_feats,
        outcome_name=target_name
    )

    cfe_data = pd.read_csv(os.path.join(root_dir, file_name))
    combined_data = pd.concat([original_sample, cfe_data], ignore_index=True)
    X_combined = combined_data.drop(columns=[target_name])
    y_combined = combined_data[target_name]

    combined_data_class = dice_ml_x.Data(
        dataframe=combined_data,
        continuous_features=orig_cont_feats,
        outcome_name=target_name
    )

    combined_data_ohe = combined_data_class.get_ohe_min_max_normalized_data(X_combined)
    

    tsne = TSNE(n_components=2, perplexity=30, n_iter=1000, random_state=42)
    combined_2d = tsne.fit_transform(combined_data_ohe)

    orig_len = len(original_sample)
    orig_sample = combined_2d[:orig_len]
    cfe_data = combined_2d[orig_len:]
    y_orig = y_combined[:orig_len]
    y_cfe = y_combined[orig_len:]
    
    y_orig = np.where(y_orig < 0.5, 0, 1)
    y_cfe = np.where(y_cfe < 0.5, 0, 1)

    unique_classes = np.unique(y_orig)

    for class_val in unique_classes:
        mask = (y_orig == class_val)
        ax.scatter(
            orig_sample[mask, 0],
            orig_sample[mask, 1],
            alpha=0.5,
            marker='v',
            label=f'Original Data - Class {class_val}'
        )

    unique_classes_cfe = np.unique(y_cfe)

    for class_val in unique_classes_cfe:
        mask = (y_cfe == class_val)
        ax.scatter(
        cfe_data[mask, 0],
        cfe_data[mask, 1],
        alpha=0.5,
        marker='x',
        label=f'CF Data - Class {class_val}'
    )

    ax.set_title(f"{dataset_name} - {backend_name}", fontsize=10)
    ax.set_xlabel('t-SNE Component 1')
    ax.set_ylabel('t-SNE Component 2')

    if all_handles is None and all_labels is None:
        all_handles, all_labels = ax.get_legend_handles_labels()

if all_handles is not None and all_labels is not None:
    fig.legend(
        all_handles, all_labels,
        loc='lower center',
        ncol=2,
        bbox_to_anchor=(0.5, 0.01)
    )

plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()



3D t-SNE plot

In [ ]:
from sklearn.manifold import TSNE

dataset_to_col = {
    "compas-recidivism": 0,
    "adult-income": 1,
    "lending-club": 2,
    "german-credit": 3
}

backend_to_row = {
    "sklearn": 0,
    "PYT": 1,
    "TF2": 2
}


all_datasets = {
    "compas-recidivism": {
        "data": helpers.load_compas_dataset(),
        "target": "twoyearrecid"
    },
    "adult-income": {
        "data": helpers.load_adult_income_dataset(),
        "target": "income"
    },
    "lending-club": {
        "data": helpers.load_lending_club_dataset(),
        "target": "loan_status"
    },
    "german-credit": {
        "data": helpers.load_german_credit_dataset(),
        "target": "credit_risk"
    }
}

root_dir = 'cfe_datasets'
cfe_files_list = os.listdir(root_dir)

fig, axes = plt.subplots(nrows=3, ncols=4, figsize=(16, 12),
                         subplot_kw=dict(projection="3d"))

fig.set_constrained_layout_pads(w_pad=5.0, h_pad=1.0, wspace=0.2, hspace=0.2)

all_handles, all_labels = None, None

for file_name in cfe_files_list:
    
    parts = file_name.split('_')
    dataset_name = parts[0]
    backend_name = parts[1].replace('.csv', '')

    row = backend_to_row[backend_name]
    col = dataset_to_col[dataset_name]
    ax = axes[row, col]

    original_data = all_datasets[dataset_name]["data"]
    target_name = all_datasets[dataset_name]["target"]

    n = 500
    if dataset_name == 'german-credit':
        n = 300
    _0_original_samples = original_data[original_data[target_name] < 0.5].sample(n=n)
    _1_original_samples = original_data[original_data[target_name] >= 0.5].sample(n=n)
    original_sample = pd.concat([_0_original_samples, _1_original_samples], ignore_index=True)

    orig_cont_feats = (
        original_sample
        .select_dtypes(include=[np.number])
        .columns
        .difference([target_name])
        .tolist()
    )

    orig_data_class = dice_ml_x.Data(
        dataframe=original_sample,
        continuous_features=orig_cont_feats,
        outcome_name=target_name
    )

    cfe_data = pd.read_csv(os.path.join(root_dir, file_name))
    combined_data = pd.concat([original_sample, cfe_data], ignore_index=True)
    X_combined = combined_data.drop(columns=[target_name])
    y_combined = combined_data[target_name]

    combined_data_class = dice_ml_x.Data(
        dataframe=combined_data,
        continuous_features=orig_cont_feats,
        outcome_name=target_name
    )

    combined_data_ohe = combined_data_class.get_ohe_min_max_normalized_data(X_combined)
    

    tsne_3d = TSNE(n_components=3, perplexity=30, n_iter=1000, random_state=42)
    combined_3d = tsne_3d.fit_transform(combined_data_ohe)

    orig_len = len(original_sample)
    orig_sample = combined_3d[:orig_len]
    cfe_data = combined_3d[orig_len:]
    y_orig = y_combined[:orig_len]
    y_cfe = y_combined[orig_len:]
    
    y_orig = np.where(y_orig < 0.5, 0, 1)
    y_cfe = np.where(y_cfe < 0.5, 0, 1)

    unique_classes = np.unique(y_orig)

    for class_val in unique_classes:
        mask = (y_orig == class_val)
        ax.scatter(
            orig_sample[mask, 0],
            orig_sample[mask, 1],
            orig_sample[mask, 2],
            alpha=0.5,
            marker='v',
            label=f'Original Data - Class {class_val}'
        )

    unique_classes_cfe = np.unique(y_cfe)

    for class_val in unique_classes_cfe:
        mask = (y_cfe == class_val)
        ax.scatter(
        cfe_data[mask, 0],
        cfe_data[mask, 1],
        cfe_data[mask, 2],
        alpha=0.5,
        marker='x',
        label=f'CF Data - Class {class_val}'
    )

    #ax.set_title(f"{dataset_name} - {backend_name}", fontsize=10)
    ax.set_xlabel('t-SNE Component 1')
    ax.set_ylabel('t-SNE Component 2')
    ax.set_zlabel('t-SNE Component 3')
    ax.set_box_aspect(None, zoom=0.90)
    if all_handles is None and all_labels is None:
        all_handles, all_labels = ax.get_legend_handles_labels()

if all_handles is not None and all_labels is not None:
    fig.legend(
        all_handles, all_labels,
        loc='lower center',
        ncol=2,
        bbox_to_anchor=(0.5, -0.01)
    )

for dataset_name, col_idx in dataset_to_col.items():
    axes[0, col_idx].set_title(dataset_name, fontsize=10)

for backend_name, row_idx in backend_to_row.items():
    axes[row_idx, 0].text2D(
        x=-0.05, y=0.5, s=backend_name,
        rotation=90,
        transform=axes[row_idx, 0].transAxes,
        ha='center', va='center',
        fontsize=10
    )

#plt.tight_layout(rect=[-0.05, 0.1, 1.2, 1])
root_folder = 'figure_artefacts'
if not os.path.exists(root_folder):
    os.makedirs(root_folder)
fig_file_path = os.path.join(root_folder, 'tsne_3d_dice_x.eps')
plt.savefig(fig_file_path, format='eps')
plt.show()

Umap of the same plot is given below.

In [ ]:
import umap.umap_ as umap

dataset_to_col = {
    "compas-recidivism": 0,
    "adult-income": 1,
    "lending-club": 2,
    "german-credit": 3
}

backend_to_row = {
    "sklearn": 0,
    "PYT": 1,
    "TF2": 2
}


all_datasets = {
    "compas-recidivism": {
        "data": helpers.load_compas_dataset(),
        "target": "twoyearrecid"
    },
    "adult-income": {
        "data": helpers.load_adult_income_dataset(),
        "target": "income"
    },
    "lending-club": {
        "data": helpers.load_lending_club_dataset(),
        "target": "loan_status"
    },
    "german-credit": {
        "data": helpers.load_german_credit_dataset(),
        "target": "credit_risk"
    }
}

root_dir = 'cfe_datasets'
cfe_files_list = os.listdir(root_dir)

fig, axes = plt.subplots(nrows=3, ncols=4, figsize=(16, 12))


all_handles, all_labels = None, None

for file_name in cfe_files_list:
    
    parts = file_name.split('_')
    dataset_name = parts[0]
    backend_name = parts[1].replace('.csv', '')

    row = backend_to_row[backend_name]
    col = dataset_to_col[dataset_name]
    ax = axes[row, col]

    original_data = all_datasets[dataset_name]["data"]
    target_name = all_datasets[dataset_name]["target"]

    n = 500
    if dataset_name == 'german-credit':
        n = 300
    _0_original_samples = original_data[original_data[target_name] < 0.5].sample(n=n)
    _1_original_samples = original_data[original_data[target_name] >= 0.5].sample(n=n)
    original_sample = pd.concat([_0_original_samples, _1_original_samples], ignore_index=True)

    orig_cont_feats = (
        original_sample
        .select_dtypes(include=[np.number])
        .columns
        .difference([target_name])
        .tolist()
    )

    orig_data_class = dice_ml_x.Data(
        dataframe=original_sample,
        continuous_features=orig_cont_feats,
        outcome_name=target_name
    )

    cfe_data = pd.read_csv(os.path.join(root_dir, file_name))
    combined_data = pd.concat([original_sample, cfe_data], ignore_index=True)
    X_combined = combined_data.drop(columns=[target_name])
    y_combined = combined_data[target_name]

    combined_data_class = dice_ml_x.Data(
        dataframe=combined_data,
        continuous_features=orig_cont_feats,
        outcome_name=target_name
    )

    combined_data_ohe = combined_data_class.get_ohe_min_max_normalized_data(X_combined)
    

    reducer = umap.UMAP(n_components=2, n_neighbors=15, random_state=42)
    combined_2d = reducer.fit_transform(combined_data_ohe)

    orig_len = len(original_sample)
    orig_sample = combined_2d[:orig_len]
    cfe_data = combined_2d[orig_len:]
    y_orig = y_combined[:orig_len]
    y_cfe = y_combined[orig_len:]
    
    y_orig = np.where(y_orig < 0.5, 0, 1)
    y_cfe = np.where(y_cfe < 0.5, 0, 1)

    unique_classes = np.unique(y_orig)

    for class_val in unique_classes:
        mask = (y_orig == class_val)
        ax.scatter(
            orig_sample[mask, 0],
            orig_sample[mask, 1],
            alpha=0.5,
            marker='v',
            label=f'Original Data - Class {class_val}'
        )

    unique_classes_cfe = np.unique(y_cfe)

    for class_val in unique_classes_cfe:
        mask = (y_cfe == class_val)
        ax.scatter(
        cfe_data[mask, 0],
        cfe_data[mask, 1],
        alpha=0.5,
        marker='x',
        label=f'CF Data - Class {class_val}'
    )

    ax.set_title(f"{dataset_name} - {backend_name}", fontsize=10)
    ax.set_xlabel('Umap Component 1')
    ax.set_ylabel('Umap Component 2')

    if all_handles is None and all_labels is None:
        all_handles, all_labels = ax.get_legend_handles_labels()

if all_handles is not None and all_labels is not None:
    fig.legend(
        all_handles, all_labels,
        loc='lower center',
        ncol=2,
        bbox_to_anchor=(0.5, 0.01)
    )

plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()



## Models' accuracies on all four datasets

We trained two types of model with three different frameworks. Firstly, a tree model is created by using sci-kit learn framework's RandomForestClassifier with the datasets with default settings. The other two models are created with PyTorch and TensorFlow frameworks. These neural networks have two layers and the architecture is same as present in the DiCE repository as follows:

\begin{equation}
\tag{3}
Linear(number\_of\_features, 20) → ReLU → Linear(20, 1) → Sigmoid
\end{equation}

While training the neural networks following hyperparameters are used:
- Learning rate: 0.001
- Number of epochs: 10
- Train dataset size: Dataset size * 80%
- Test dataset size: Dataset size * 20%
- Number of batches for training dataset: 16
- Number of batches for test dataset: 4
- Optimizer: Adam

In [ ]:
benchmarking_results

In [ ]:
# Accuracy table for each dataset

from IPython.display import display
import pandas as pd

accuracy_data = {
    dataset: {
        model: {
            'accuracy':  f"{round(details['model_metrics']['accuracy'] * 100, 2)}%" 
        }
        for model, details in models.items()
    }
    for dataset, models in benchmarking_results.items()
}

rows = []

for dataset, models in accuracy_data.items():
    for model_name, model_info in models.items():
        row = {
            'Dataset': dataset,
            'Model': 'Random Forest Classifier' if model_name == 'sklearn' else f'Neural Network ({model_name})',
            'Accuracy': model_info.get('accuracy'),
        }
        rows.append(row)


accuracy_df = pd.DataFrame(rows)
display(accuracy_df)
accuracy_df['Dataset'] = accuracy_df["Dataset"].mask(accuracy_df["Dataset"].duplicated(), "")
markdown_table = accuracy_df.to_markdown(index=False)
print(markdown_table)

| Dataset           | Model                    | Accuracy   |
|:------------------|:-------------------------|:-----------|
| compas-recidivism | Random Forest Classifier | 57.14%     |
|                   | Neural Network (PYT)     | 66.0%      |
|                   | Neural Network (TF2)     | 65.9%      |
| adult-income      | Random Forest Classifier | 81.97%     |
|                   | Neural Network (PYT)     | 83.46%     |
|                   | Neural Network (TF2)     | 83.14%     |
| lending-club      | Random Forest Classifier | 82.31%     |
|                   | Neural Network (PYT)     | 82.84%     |
|                   | Neural Network (TF2)     | 82.97%     |
| german-credit     | Random Forest Classifier | 71.5%      |
|                   | Neural Network (PYT)     | 79.0%      |
|                   | Neural Network (TF2)     | 76.0%      |


As it can be seen from the table the models performed satisfying both with [Adult Income Dataset](https://archive.ics.uci.edu/dataset/2/adult) and the [Lending Club Dataset](https://www.lendingclub.com/). While models perform moderately on [German Credit Risk Dataset](https://archive.ics.uci.edu/static/public/144/statlog+german+credit+data.zip), they perform poorly on [Compas Recidivism Dataset](https://api.openml.org/data/download/22111929/dataset).

## Explainers' counterfactual generation time with different datasets

Three types of explainers are generated for counterfactual generation that are genetic, PyTorch, and TensorFlow. Counterfactuals generated by genetic algorithm are generated with a genetic algorithm creates mutations with the best counterfactuals depending on the loss value. The gradient methods use Adam optimizer with a learning rate of $0.05$ and parameterize the counterfactuals to optimize counterfactuals. During the counterfactual generation process 5 counterfactuals has been created. The table below shows time spent for counterfactual generation for each model and dataset. PYT represents a neural network model created with PyTorch and TF2 represents a neural network model created with TensorFlow framework.

## Explainers' counterfactual generation time with different datasets

Three types of explainers are generated for counterfactual generation that are genetic, PyTorch, and TensorFlow. Counterfactuals generated by genetic algorithm are generated with a genetic algorithm creates mutations with the best counterfactuals depending on the loss value. The gradient methods use Adam optimizer with a learning rate of $0.05$ and parameterize the counterfactuals to optimize counterfactuals. During the counterfactual generation process 5 counterfactuals has been created. The table below shows time spent for counterfactual generation for each model and dataset. PYT represents a neural network model created with PyTorch and TF2 represents a neural network model created with TensorFlow framework.

| Dataset           | Model          |   Time (s) |
|:------------------|:---------------|------------:|
| compas-recidivism | Genetic        |       10.22 |
| compas-recidivism | Gradient (PYT) |        2.69 |
| compas-recidivism | Gradient (TF2) |       30.95 |
| adult-income      | Genetic        |       49.08 |
| adult-income      | Gradient (PYT) |        7.14 |
| adult-income      | Gradient (TF2) |       43.72 |
| lending-club      | Genetic        |      110.42 |
| lending-club      | Gradient (PYT) |       17.32 |
| lending-club      | Gradient (TF2) |      200.41 |
| german-credit     | Genetic        |      217.62 |
| german-credit     | Gradient (PYT) |       10.89 |
| german-credit     | Gradient (TF2) |       61.37 |

In [ ]:
# Counterfactual generation time table
time_spent_data = {
    dataset: {
        model: {
            'time': round(details['time'], 2)
        }
        for model, details in models.items()
    }
    for dataset, models in benchmarking_results.items()
}

rows = []

for dataset, models in time_spent_data.items():
    for model_name, model_info in models.items():
        row = {
            'Dataset': dataset,
            'Model': 'Genetic' if model_name == 'sklearn' else f'Gradient ({model_name})',
            'Time (ms)': model_info.get('time'),
        }
        rows.append(row)

time_df = pd.DataFrame(rows)
display(time_df)
markdown_table = time_df.to_markdown(index=False)
print(markdown_table)

## Metrics and Sensitivity Analysis for Dice Extended


### 1. Robustness Metrics

#### Dice-Sørensen Coefficient

To evaluate robustness, the Dice-Sørensen coefficient measures the similarity between counterfactuals c1 and
c2 generated for similar input instances x1 and x2:

\begin{equation}
\tag{4}
Robustness(c_1, c_2) = \frac{2 * \lvert c_1 \cap c_2 \rvert}{\lvert c_1 \rvert + \lvert c_2 \rvert}
\end{equation}

where:
- $ c_1 $ and $ c_2 $ are binary vectors,
- $ \lvert c_1 \cap c_2 \rvert $: The number of shared (overlapping) features between c1 and c2,
- $ \lvert c_1 \rvert $ and $ \lvert c_2 \rvert $: The total number of features in each counterfactual.

#### Input Perturbation and Stability

Stability under input perturbation measures the solution variance when slight perturbations are introduced
to the input instance. The procedure includes the following steps:

1) **Apply Gaussian Noise:** Perturb the input $x$ by adding Gaussian noise $\delta$ to create perturbed inputs
$x'$:

\begin{equation}
\tag{5}
x' = x + \delta, \quad \delta \sim \mathcal{N}(0, \sigma^2)
\end{equation}

where $\sigma$ is the standard deviation of the noise (e.g., $\sigma = 0.01$).

2) **Generate Counterfactuals:** Generate counterfactual explanations $c_i$ for the original input $x$ and $c_i'$ for the perturbed input $x'$.

3) **Measure Stability:** Compare counterfactuals using a distance metric, such as the Euclidean distance:

\begin{equation}
\tag{6}
Stability = \frac{1}{n} \sum_{i=1}^{n} dist(c_i, c_i')
\end{equation}

where:

\begin{equation}
\tag{7}
dist(c_i, c_i') = \sqrt{\sum_{j=1}^{d} (c_{ij} - c_{ij}')^2}
\end{equation}

$n$ is the total number of input instances, $c_i$ is the counterfactual for the original input, and $c_i'$ is the counterfactual for the perturbed input.

In the section below we will do the required computation to calculate the stability of the counterfactuals. Firstly we pick an instance from the dataset which is $x$ and generate counterfactuals to it. Subsequently, we perturb $x$ and obtain $x'$ and generate counterfactuals also for it.

In [ ]:
with open('benchmarking_final_results.pkl', 'rb') as res_file:
    benchmarking_results = pickle.load(res_file)

In [ ]:
all_datasets = {
    "compas-recidivism": {
        "data": helpers.load_compas_dataset(),
        "target": "twoyearrecid"
    },
    "adult-income": {
        "data": helpers.load_adult_income_dataset(),
        "target": "income"
    },
    "lending-club": {
        "data": helpers.load_lending_club_dataset(),
        "target": "loan_status"
    },
    "german-credit": {
        "data": helpers.load_german_credit_dataset(),
        "target": "credit_risk"
    }
}



cfes_for_metrics = {'adult-income': {}, 'lending-club': {},
                    'german-credit': {}, 'compas-recidivism': {}}

for df_name, df_dict in all_datasets.items():
    df = df_dict['data']
    target = df_dict['target']
    target_col = df[target]
    train_dataset, test_dataset, y_train, y_test =train_test_split(df, target_col,
                                                               test_size=0.2, random_state=42,
                                                               stratify=target_col)
    

    dummy_state_dict = torch.load(benchmarking_results[df_name]['PYT']['model_path'])
    dummy_state_dict = {f'model.{key}': value for key, value in dummy_state_dict.items()}
    in_features = dummy_state_dict['model.0.weight'].shape[1]
    model_path = benchmarking_results[df_name]['PYT']['model_path']
    model = neuralnetworks.PYTModel(in_features)
    model.load_state_dict(dummy_state_dict)

    cont_feats = df.select_dtypes(include=[np.number]).columns.difference([target]).tolist()
    d = dice_ml_x.Data(dataframe=train_dataset, continuous_features=cont_feats, outcome_name=target)
    gaussian_kwargs = {}
    m = dice_ml_x.Model(model=model.model, backend='PYT', func='ohe-min-max')
    exp = dice_ml_x.Dice(d, m, method="gradient")

    x_train, x_test = train_dataset.drop(columns=[target]), test_dataset.drop(columns=[target])

    x = x_test[1:2]

    dice_exp = exp.generate_counterfactuals(x, total_CFs=10, desired_class="opposite",
                                            perturbation_method="gaussian", proximity_weight=0.0,
                                            diversity_weight=0.2,
                                            robustness_weight=0.8,
                                            **gaussian_kwargs)

    
    C = dice_exp.to_dataframe()
    cfes_for_metrics[df_name]['cfe_object'] = dice_exp
    cfes_for_metrics[df_name]['C'] = C
    cfes_for_metrics[df_name]['x'] = x
    cfes_for_metrics[df_name]['m'] = m
    cfes_for_metrics[df_name]['d'] = d
    cfes_for_metrics[df_name]['target'] = target
    cfes_for_metrics[df_name]['x_with_target'] = test_dataset[1:2]

In [ ]:
from IPython.display import display

for df_name, cfe_dict in cfes_for_metrics.items():
    x = cfe_dict['x']
    C = cfe_dict['C']
    cfe_object = cfe_dict['cfe_object']
    print(f'-------------------------------------{df_name}-------------------------------------')
    cfe_object.visualize_as_dataframe(show_only_changes=True)
    display(C)

In [ ]:
def do_perturbation(x: pd.DataFrame, data_class):
    x_tensor = torch.tensor(x.values, dtype=torch.float32)
    
    #cfs_perturbed = torch.nn.Parameter(x_tensor.clone(), requires_grad=True)
    cat_cols = data_class.get_encoded_categorical_feature_indexes()
    cat_cols = [col for group in cat_cols for col in group]
    continuous_feature_indexes = list(set(list(range(len(x.columns)))) - set(cat_cols))
    categorical_feature_indexes = data_class.get_encoded_categorical_feature_indexes()
    if continuous_feature_indexes:
        
        continuous_slice = x_tensor[:, continuous_feature_indexes]
        noise = continuous_slice * 0.1
        noise_mask = torch.zeros_like(x_tensor)
        noise_mask[:, continuous_feature_indexes] = noise
        x_tensor = x_tensor + noise_mask

    if categorical_feature_indexes:
        for cat_cols in categorical_feature_indexes:
            cat_slice = x_tensor[:, cat_cols]
            sample_size = cat_slice.shape[0]
            num_cats = cat_slice.shape[1]

            rand_idx = torch.randint(low=0, high=num_cats, size=(sample_size, ))

            cat_slice_perturbed = torch.nn.functional.one_hot(rand_idx, num_classes=num_cats).float()

            cat_mask = torch.zeros_like(x_tensor)
            cat_mask[:, cat_cols] = cat_slice_perturbed
            x_perturbed = x_tensor + cat_mask
    return x_perturbed

def generate_perturbations(x_ohe: pd.DataFrame, model: any, data_class: dice_ml_x.Data,
                           max_iter=100, tol=1e-3, gamma=1e-2):
        x_ohe_tensor = torch.tensor(x_ohe.values, dtype=torch.float32, requires_grad=True)
        x_perturbed = do_perturbation(x_ohe, data_class)
        perturbation_optimizer = torch.optim.Adam([x_perturbed], lr=1e-3)

        prev_loss = np.inf
        for _ in range(max_iter):
            with torch.no_grad():
                model.model.eval()
                pred_i = model.model(x_ohe_tensor)
                pred_i_prime = model.model(x_perturbed)
            class_loss = torch.mean((pred_i - pred_i_prime) ** 2)
            distance = torch.norm(x_perturbed - x_ohe_tensor, p=2)
            loss = class_loss + gamma * distance

            perturbation_optimizer.zero_grad()
            loss.backward()

            perturbation_optimizer.step()
            if abs(loss.item() - prev_loss) < tol:
                break
            prev_loss = loss.item()
        return x_perturbed.detach()

In [ ]:
for df_name, cfe_dict in cfes_for_metrics.items():
    data_class = cfe_dict['d']
    model_object = cfe_dict['m']
    x = cfe_dict['x']
    C = cfe_dict['C']
    target = cfe_dict['target']
    x_ohe = data_class.get_ohe_min_max_normalized_data(x)
    x_prime_ohe_tensor = generate_perturbations(x_ohe, model_object, data_class)
    x_prime_decoded = data_class.get_decoded_data(x_prime_ohe_tensor.numpy())
    x_prime = data_class.get_inverse_ohe_min_max_normalized_data(x_prime_decoded)

    exp = dice_ml_x.Dice(data_class, model_object, method="gradient")
    dice_exp_prime = exp.generate_counterfactuals(x_prime, total_CFs=10, desired_class="opposite",
                                            perturbation_method="gaussian", **gaussian_kwargs)
    C_prime = dice_exp_prime.to_dataframe()
    na_cols = C_prime.columns[C_prime.isna().any().tolist()].tolist()
    C_prime[na_cols] = C_prime[na_cols].fillna(x_prime[na_cols].iloc[0])
    C_prime[target] = (C_prime[target] >= 0.5).astype(int)
    cfes_for_metrics[df_name]['C_prime'] = C_prime


In [ ]:
def compute_stability(C_set_1: torch.Tensor, C_set_2: torch.Tensor, p: int=2) -> float:
    return torch.mean(torch.cdist(C_set_1, C_set_2, p=2)).item()

In [ ]:
for df_name, cfe_dict in cfes_for_metrics.items():
    data_class = cfe_dict['d']
    C = cfe_dict['C']
    C_prime = cfe_dict['C_prime']
    C_ohe_tensor = torch.tensor(data_class.get_ohe_min_max_normalized_data(C).values, dtype=torch.float32)
    C_prime_ohe_tensor = torch.tensor(data_class.get_ohe_min_max_normalized_data(C_prime).values, dtype=torch.float32)

    stability = round(compute_stability(C_ohe_tensor, C_prime_ohe_tensor), 2)
    stability_max = round(torch.sqrt(torch.tensor(C_ohe_tensor.shape[1], dtype=torch.int16)).item(), 2)
    cfes_for_metrics[df_name]['stability'] = round(stability, 2)
    cfes_for_metrics[df_name]['stability_max'] = round(stability_max, 2)
    print(f"Stability for the dataset {df_name} is {stability}/{stability_max}")


In [ ]:
cfes_for_metrics

#### Result for Stability

We computed the stability metric by converting the $C$ that is a counterfactuals set which has 10 counterfactual samples and $C'$ is the counterfactuals set that has the same number of counterfactuals as $C$ that is generated with $x'$ which is perturbed original instance. Firstly, $C$ is converted into a normalized vector that are of shape (10, n) which means there are 10 samples with n features. The result of the calculation of stability with the maximum possible value for each dataset under input perturbation is given below.

| Dataset (Original) | Stability | Stability (max) | #features (one-hot) |
|--------------------|-----------|-----------------|---------------------|
| adult-income       | 2.53      | 5.48            | 30                  |
| lending-club       | 2.26      | 8.37            | 70                  |
| german-credit      | 3.43      | 7.87            | 62                  |
| adult-income       | 1.75      | 3.0             | 9                   |


### 2. Counterfactual Quality Measures

#### Fidelity

Fidelity measures how often generated counterfactuals successfully change the model’s prediction:

\begin{equation}
\tag{9}
Fidelity = \frac{\sum_{i=1}^{n} \mathbf{1}(f(c_i) = y_{desired})}{n}
\end{equation}

where:

- $f$: Prediction model,
- $c_i$: Counterfactual instance,
- $y_{desired}$: Target output class,
- $n$: Total number of counterfactuals.

Since all generated counterfactuals belong to the desired class $fidelity$ of the counterfactuals is $100\%$

#### Proximity

Proximity measures the average distance between counterfactuals $c_i$ and the original inputs $x_i$:

\begin{equation}
\tag{10}
Proximity = \frac{1}{n} \sum_{i=1}^{n} dist(x_i, c_i)
\end{equation}

The Manhattan distance can be used for simplicity:

\begin{equation}
\tag{11}
dist(x_i, c_i) = \sum_{j=1}^{d} \lvert x_{ij} - c_{ij} \rvert
\end{equation}


In [ ]:
def compute_proximity(original_instance: torch.Tensor, C: torch.Tensor) -> float:
    return torch.mean(torch.cdist(original_instance, C, p=1)).item()

In [ ]:
for df_name, cfe_dict in cfes_for_metrics.items():
    data_class = cfe_dict['d']
    target = cfe_dict['target']
    x = cfe_dict['x_with_target']
    C = cfe_dict['C']
    C_ohe_tensor = torch.tensor(data_class.get_ohe_min_max_normalized_data(C).values, dtype=torch.float32)
    x_ohe_tensor = torch.tensor(data_class.get_ohe_min_max_normalized_data(x).values, dtype=torch.float32)

    proximity = round(compute_proximity(x_ohe_tensor, C_ohe_tensor), 2)
    proximity_max = C_ohe_tensor.shape[1]
    cfes_for_metrics[df_name]['proximity'] = proximity
    cfes_for_metrics[df_name]['proximity_max'] = proximity_max
    print(f"Proximity for {df_name} is {proximity}/{proximity_max}")

#### Result for Proximity

We computed the proximity of $x$ and $C$ we converted them into normalized and one hot encoded tensors of shape (10, n). The resulting total proximity between the original instance and generated counterfactuals is given below in the table which is highly acceptable when we consider the maximum Manhattan distance between these two tensors.

| Dataset (Original) | Proximity | Proximity (max) | #features (one-hot) |
|--------------------|-----------|-----------------|---------------------|
| adult-income       | 5.99      | 30              | 30                  |
| lending-club       | 6.03      | 70              | 70                  |
| german-credit      | 11.69     | 62              | 62                  |
| compas-recidivism  | 3.84      | 9               | 9                   |


#### Diversity

Diversity measures how dissimilar the counterfactuals $c_1, c_2, c_3,\ldots,c_k$ are among themselves:

\begin{equation}
\tag{12}
Diversity = \frac{1}{k(k-1)}\sum_{i_1}^{k}\sum_{j \neq i}^{} dist(c_i, c_j)
\end{equation}

where $k$ is the number of counterfactuals.

In [ ]:
def compute_diversity(C_ohe: torch.Tensor):
    k = C_ohe.shape[0]
    pairwise_dist = torch.cdist(C_ohe, C_ohe, p=2)
    diversity = (torch.sum(pairwise_dist) - torch.sum(torch.diagonal(pairwise_dist))) / (k * (k - 1))
    return diversity.item()

In [ ]:
import math
for df_name, cfe_dict in cfes_for_metrics.items():
    C = cfe_dict['C']
    data_class = cfe_dict['d']
    n = len(C)
    k = 2

    C_ohe_tensor = torch.tensor(data_class.get_ohe_min_max_normalized_data(C).values, dtype=torch.float32)
    diversity = round(compute_diversity(C_ohe_tensor), 2)
    diversity_max = round(math.sqrt(C_ohe_tensor.shape[1]), 2)
    cfes_for_metrics[df_name]['diversity'] = diversity
    cfes_for_metrics[df_name]['diversity_max'] = diversity_max
    print(f"Diversity for {df_name} is {diversity} / {diversity_max}")

#### Result for Diversity

For the counterfactuals tensor we work with minimum diversity value is $0$ that indicates all counterfactuals are identical. For the condition that all counterfactuals are distinct that makes the diversity value maximum which is $\sqrt{n}$ where $n$ is the number of features of the one hot encoded tensor. The diversity value we calculated with the counterfactuals for each datasets are given in the table below which seems that counterfactuals moderately spread in the features space.

| Dataset (Original) | Diversity | Diversity (max) | #features (one-hot) |
|--------------------|-----------|-----------------|---------------------|
| adult-income       | 2.66      | 5.48            | 30                  |
| lending-club       | 1.75      | 8.37            | 70                  |
| german-credit      | 3.52      | 7.87            | 62                  |
| adult-income       | 1.66      | 3.0             | 9                   |

In [ ]:
#with open('cfes_for_metrics.pkl', 'wb') as cfm_file:
#    pickle.dump(cfes_for_metrics, cfm_file)
with open('cfes_for_metrics.pkl', 'rb') as cfm_file:
    cfes_for_metrics = pickle.load(cfm_file)

### 3. Sensitivity Analysis

#### Objective Function with Weights

The modified loss function in DiCE-Extended is defined as in the [equation 1](#equation-1) where:

- $yloss(f(c_i), y)$: Prediction loss for counterfactual instance $c_i$ relative to the desired outcome $y$,
- $dist(c_i, x)$: Distance metric (e.g., Euclidean or Manhattan) between the counterfactual c_i and the original input $x$,
- $dpp\_diversity(c_1,\ldots,c_k)$: Diversity loss term based on Determinantal Point Process (DPP),
- $Robustness(c_i,c_i')$: Robustness loss measuring similarity of counterfactuals under perturbations.

  The weights $\lambda_1, \lambda_2, \lambda_3$ control the balance between proximity, diversity, and robustness, respectively.

#### Sensitivity Analysis

To perform sensitivity analysis:

1) Vary the weights $\lambda_1, \lambda_2, \lambda_3$ systematically while ensuring:

\begin{equation}
\tag{13}
\lambda_1 + \lambda_2 + \lambda_3 = 1 \text{ (for normalization)}.
\end{equation}

2) Track the changes in the following metrics:

\begin{equation}
\tag{14}
P(\lambda_1, \lambda_2, \lambda_3) = Proximity,
\end{equation}

\begin{equation}
\tag{15}
D(\lambda_1, \lambda_2, \lambda_3) = Diversity,
\end{equation}

\begin{equation}
\tag{16}
R(\lambda_1, \lambda_2, \lambda_3) = Robustness,
\end{equation}

3) Measure the relationship between these metrics and the weights.

In [ ]:
weight_grid = [(l1, l2, 1 - l1 - l2)
               for l1, l2 in itertools.product(np.linspace(0, 1, 11), repeat=2)
               if l1 + l2 <= 1]

grid_search_df = helpers.load_adult_income_dataset()
target_col = grid_search_df['income']
train_dataset, test_dataset, y_train, y_test = train_test_split(grid_search_df, target_col,
                                                               test_size=0.2, random_state=42,
                                                               stratify=target_col)

dummy_state_dict = torch.load(benchmarking_results['adult-income']['PYT']['model_path'])
dummy_state_dict = {f'model.{key}': value for key, value in dummy_state_dict.items()}
in_features = dummy_state_dict['model.0.weight'].shape[1]
model_path = benchmarking_results['adult-income']['PYT']['model_path']
model = neuralnetworks.PYTModel(in_features)
model.load_state_dict(dummy_state_dict)

cont_feats = grid_search_df.select_dtypes(include=[np.number]).columns.difference(['income']).tolist()
d = dice_ml_x.Data(dataframe=train_dataset, continuous_features=cont_feats, outcome_name='income')
gaussian_kwargs = {}
m = dice_ml_x.Model(model=model.model, backend='PYT', func='ohe-min-max')
exp = dice_ml_x.Dice(d, m, method="gradient")

x_train, x_test = train_dataset.drop(columns=['income']), test_dataset.drop(columns=['income'])

x = x_test[1:2]
x_ohe = d.get_ohe_min_max_normalized_data(test_dataset[1:2])
x_ohe_tensor = torch.tensor(x_ohe.values, dtype=torch.float32)
results = []
for proximity_weight, diversity_weight, robustness_weight in tqdm(weight_grid, desc="Generating Counterfactuals"):
    cf_explanations = exp.generate_counterfactuals(x, total_CFs=10, desired_class="opposite",
                                                   proximity_weight=proximity_weight,
                                                   diversity_weight=diversity_weight,
                                                   robustness_weight=robustness_weight,
                                                   **gaussian_kwargs)
    proximity_loss = exp.loss_history['proximity_loss'][-1]
    diversity_loss = exp.loss_history['diversity_loss'][-1]
    robustness_loss = exp.loss_history['robustness_loss'][-1]
    total_loss = exp.loss_history['total_loss'][-1]
    C = cf_explanations.to_dataframe()
    C_ohe = d.get_ohe_min_max_normalized_data(C)
    C_ohe_tensor = torch.tensor(C_ohe.values, dtype=torch.float32)
    
    proximity = compute_proximity(x_ohe_tensor, C_ohe_tensor)
    diversity = compute_diversity(C_ohe_tensor)
    C_targetless = C.drop(columns=['income'])
    C_ohe_targetless = d.get_ohe_min_max_normalized_data(C_targetless)
    C_ohe_prime_tensor = generate_perturbations(C_ohe_targetless, model, d)
    robustness = torch.mean(torch.norm(torch.tensor(C_ohe_targetless.values, dtype=torch.float32) - C_ohe_prime_tensor, dim=1)).item()
    results.append((proximity_weight, diversity_weight, robustness_weight, proximity,
                    diversity, robustness, proximity_loss, diversity_loss, robustness_loss,
                    total_loss))

results = np.array(results)

In [ ]:
#with open('weight_sensitivity_results.pkl', 'wb') as sens_file:
#    pickle.dump(results, sens_file)

In [ ]:
with open('weight_sensitivity_results.pkl', 'rb') as sens_file:
    sens_res_loaded = pickle.load(sens_file)

In [ ]:
sens_res_loaded

In [ ]:
def aggregated_objective(row):
    return row[3] - row[4] - row[5]

n_bootstraps = 1000
optimal_weights_bootstrap = []

for b in range(n_bootstraps):
    # Sample with replacement from the data.
    indices = np.random.choice(len(sens_res_loaded), size=len(sens_res_loaded), replace=True)
    boot_data = sens_res_loaded[indices]
    
    # Compute the objective for each row in the bootstrap sample.
    objectives = np.apply_along_axis(aggregated_objective, 1, boot_data)
    
    # Select the index with the minimum objective.
    best_idx = np.argmin(objectives)
    
    # Record the corresponding weight combination (columns 0, 1, 2).
    optimal_weights_bootstrap.append(boot_data[best_idx, :3])

optimal_weights_bootstrap = np.array(optimal_weights_bootstrap)

# Compute 95% confidence intervals for each weight.
ci_lower = np.percentile(optimal_weights_bootstrap, 2.5, axis=0)
ci_upper = np.percentile(optimal_weights_bootstrap, 97.5, axis=0)

print("95% Confidence Intervals for the Weights:")
print("Proximity weight: {:.3f} - {:.3f}".format(ci_lower[0], ci_upper[0]))
print("Diversity weight: {:.3f} - {:.3f}".format(ci_lower[1], ci_upper[1]))
print("Robustness weight: {:.3f} - {:.3f}".format(ci_lower[2], ci_upper[2]))

In [ ]:
sens_res_loaded

In [ ]:
from mpl_toolkits.axes_grid1 import host_subplot
import mpl_toolkits.axisartist as AA

data = sens_res_loaded
lambda3 = data[:, 2]
robust_loss = data[:, 5]
total_loss = data[:, 9]

# Create a host subplot that supports multiple y-axes.
host = host_subplot(111, axes_class=AA.Axes)
plt.subplots_adjust(right=0.75)  # Reserve space on right for extra y-axes.

# Create twin axes for additional y-axes.
par1 = host.twinx()  # For diversity loss.
par2 = host.twinx()  # For proximity loss.

# Offset the third y-axis (par2) to the right.
offset = 60  # Adjust offset as needed.
new_fixed_axis = par2.get_grid_helper().new_fixed_axis
par2.axis["right"] = new_fixed_axis(loc="right", axes=par2, offset=(offset, 0))
par2.axis["right"].toggle(all=True)

# Set axis labels.
host.set_xlabel(r'$\lambda_3$', fontsize=12)
host.set_ylabel('Robustness Loss', color='blue', fontsize=12)
par2.set_ylabel('Total Loss', color='green', fontsize=12)

# Plot scatter points.
p1 = host.scatter(lambda3, robust_loss, color='blue', marker='o', s=100, label='Robustness Loss')
p2 = par1.scatter(lambda3, total_loss, color='green', marker='s', s=100, label='Total Loss')


# Connect the dots with line segments.
def connect_dots(x, y, ax, color):
    # Sort x and y by x.
    sort_idx = np.argsort(x)
    x_sorted = x[sort_idx]
    y_sorted = y[sort_idx]
    ax.plot(x_sorted, y_sorted, color=color, linestyle='-', linewidth=1)

connect_dots(lambda3, robust_loss, host, 'blue')
connect_dots(lambda3, total_loss, par2, 'green')

# Combine legends from all axes.
handles1, labels1 = host.get_legend_handles_labels()
handles2, labels2 = par1.get_legend_handles_labels()

host.legend(handles1 + handles2, labels1, ncol=3, bbox_to_anchor=(0.5, -0.2),
            loc='lower center')

plt.title(r'$\lambda_r$ vs. Computed Robustness - Total Loss', fontsize=14)
root_folder = 'figure_artefacts'
if not os.path.exists(root_folder):
    os.makedirs(root_folder)
fig_file_path = os.path.join(root_folder, 'lambda_3_vs_losses.eps')
plt.savefig(fig_file_path, format='eps', bbox_inches="tight")
plt.show()

In [ ]:
iterations = np.arange(len(sens_res_loaded))

# Extract columns for readability
lambda1    = sens_res_loaded[:, 0]
lambda2    = sens_res_loaded[:, 1]
lambda3    = sens_res_loaded[:, 2]
proximity  = sens_res_loaded[:, 3]
diversity  = sens_res_loaded[:, 4]
robustness = sens_res_loaded[:, 5]

plt.figure(figsize=(10,6))

plt.plot(iterations, lambda1,    label='Lambda 1',    marker='o')
plt.plot(iterations, lambda2,    label='Lambda 2',    marker='o')
plt.plot(iterations, lambda3,    label='Lambda 3',    marker='o')
plt.plot(iterations, proximity,  label='Proximity',   marker='x')
plt.plot(iterations, diversity,  label='Diversity',   marker='^')
plt.plot(iterations, robustness, label='Robustness',  marker='v')

plt.xlabel("Iteration")
plt.ylabel("Value")
plt.title("Weights and Metrics over Iterations")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from kneed import KneeLocator  # make sure to install via: pip install kneed


# Compute an aggregated metric for each combination.
# Here we simply sum the computed metrics,
# but you could weight them differently if needed.
agg_metric = sens_res_loaded[:, 3] - sens_res_loaded[:, 4] - sens_res_loaded[:, 5]

# Plot aggregated metric vs iteration index
iterations = np.arange(len(agg_metric))
plt.figure(figsize=(10, 6))
plt.plot(iterations, agg_metric, marker='o')
plt.xlabel("Iteration")
plt.ylabel("Aggregated Metric")
plt.title("Elbow Method: Aggregated Metric vs. Iteration")
plt.show()

# Use KneeLocator to automatically detect the elbow (knee) in the curve.
# Here we assume the aggregated metric is convex and decreasing.
knee_locator = KneeLocator(iterations, agg_metric, curve='convex', direction='decreasing')
elbow_iter = knee_locator.knee

print("Elbow point detected at iteration:", elbow_iter)
if elbow_iter is not None:
    optimal_weights = sens_res_loaded[int(elbow_iter), :3]
    print("Optimal weight combination (proximity, diversity, robustness):", optimal_weights)
else:
    print("No clear elbow was detected.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from scipy.spatial.distance import cdist
from kneed import KneeLocator  # install via: pip install kneed


# --- Step 1: Elbow Method on 3D Computed Metrics ---
# Extract computed metrics: columns 3, 4, and 5.
X = sens_res_loaded[:, 3:6]

distortions = []
inertias = []
K = range(1, 10)
for k in K:
    kmeans = KMeans(n_clusters=k, random_state=42).fit(X)
    # Distortion: average squared distance of each point to its nearest cluster center
    distortion = sum(np.min(cdist(X, kmeans.cluster_centers_, 'euclidean'), axis=1)**2) / X.shape[0]
    distortions.append(distortion)
    inertias.append(kmeans.inertia_)

# Plot the Distortion elbow curve
plt.figure(figsize=(8, 6))
plt.plot(K, distortions, 'bx-')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Distortion')
plt.title('Elbow Method using Distortion (3D Computed Metrics)')
plt.grid(True)
plt.show()

# Plot the Inertia elbow curve
plt.figure(figsize=(8, 6))
plt.plot(K, inertias, 'bx-')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow Method using Inertia (3D Computed Metrics)')
plt.grid(True)
plt.show()

# Use KneeLocator to detect the optimal k based on the distortion curve.
knee_locator = KneeLocator(list(K), distortions, curve='convex', direction='decreasing')
optimal_k = knee_locator.knee
print("Optimal number of clusters (k) according to distortion:", optimal_k)


X3D = sens_res_loaded[:, 3:6]

k_range = range(1, 5)
for k in k_range:
    kmeans = KMeans(n_clusters=k, init='k-means++', random_state=42)
    y_kmeans = kmeans.fit_predict(X3D)
    
    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection='3d')
    sc = ax.scatter(X3D[:, 0], X3D[:, 1], X3D[:, 2], c=y_kmeans, cmap='viridis',
                    marker='o', edgecolor='k', s=100)
    ax.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
               kmeans.cluster_centers_[:, 2],
               s=300, c='red', label='Centroids', edgecolor='k')
    ax.set_title(f'3D K-means Clustering (k={k})')
    ax.set_xlabel('Computed Proximity')
    ax.set_ylabel('Computed Diversity')
    ax.set_zlabel('Computed Robustness')
    ax.legend()
    plt.show()

# --- Step 3: (Optional) Selecting Optimal Weight Combinations ---
# We define an aggregated objective that mirrors our loss: 
# Objective = computed proximity - computed diversity - computed robustness.
objective = sens_res_loaded[:, 3] - sens_res_loaded[:, 4] - sens_res_loaded[:, 5]

# For each cluster (from optimal_k), select the grid search point (row) that minimizes the objective.
kmeans_opt = KMeans(n_clusters=optimal_k, random_state=42).fit(X)
cluster_labels = kmeans_opt.labels_

best_indices = []
for cl in range(optimal_k):
    indices = np.where(cluster_labels == cl)[0]
    if len(indices) > 0:
        cluster_objectives = objective[indices]
        best_idx_in_cluster = indices[np.argmin(cluster_objectives)]
        best_indices.append(best_idx_in_cluster)

best_weight_combinations = sens_res_loaded[best_indices, :3]
print("Optimal weight combinations (one per cluster):")
print(best_weight_combinations)


In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # For 3D plotting
from sklearn.cluster import KMeans

# Assume X is a NumPy array of shape (N, 3) containing:
# [computed_proximity, computed_diversity, computed_robustness]
# For example:
# X = data[:, 3:6]

# Initialize a range of k values
k_range = range(1, 5)

for k in k_range:
    kmeans = KMeans(n_clusters=k, init='k-means++', random_state=42)
    y_kmeans = kmeans.fit_predict(X)
    
    # Create a new figure with 3D axes
    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection='3d')
    
    # Plot the clustered data points in 3D
    scatter = ax.scatter(
        X[:, 0], X[:, 1], X[:, 2],
        c=y_kmeans, cmap='viridis', marker='o', edgecolor='k', s=100
    )
    # Plot the cluster centers in 3D
    ax.scatter(
        kmeans.cluster_centers_[:, 0], 
        kmeans.cluster_centers_[:, 1], 
        kmeans.cluster_centers_[:, 2],
        s=300, c='red', label='Centroids', edgecolor='k'
    )
    
    ax.set_title(f'K-means Clustering (k={k})')
    ax.set_xlabel('Computed Proximity')
    ax.set_ylabel('Computed Diversity')
    ax.set_zlabel('Computed Robustness')
    ax.legend()
    plt.show()

In [ ]:
objective = sens_res_loaded[:, 3] - sens_res_loaded[:, 4] - sens_res_loaded[:, 5]

# Find the index of the best (minimum) objective value
best_idx = np.argmin(objective)
best_weights = sens_res_loaded[best_idx, :3]       # [lambda1, lambda2, lambda3]
best_metrics = sens_res_loaded[best_idx, 3:]         # [computed proximity, diversity, robustness]

print("Best weight combination (lambda1, lambda2, lambda3):", best_weights)
print("Corresponding computed metrics (proximity, diversity, robustness):", best_metrics)
print("Aggregated objective value:", objective[best_idx])

# Plot the 3D scatter of computed metrics vs. weight combinations.
# We'll plot computed proximity, diversity, and robustness in 3D.
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')

# Use a color mapping for the objective
sc = ax.scatter(sens_res_loaded[:, 3], sens_res_loaded[:, 4], sens_res_loaded[:, 5], c=objective, cmap='viridis', marker='o', s=100)

# Highlight the best weight combination
ax.scatter(best_metrics[0], best_metrics[1], best_metrics[2],
           c='red', s=200, label='Best Weights')

ax.set_xlabel('Computed Proximity')
ax.set_ylabel('Computed Diversity')
ax.set_zlabel('Computed Robustness')
ax.set_title('Grid Search: Computed Metrics vs. Weight Combinations')
ax.legend()
plt.colorbar(sc, label='Aggregated Objective')
plt.show()

In [ ]:
columns = ['proximity_weight', 'diversity_weight', 'robustness_weight', 'proximity', 'diversity', 'robustness']
results_df = pd.DataFrame(results, columns=columns)

root_folder = 'figure_artefacts'
if not os.path.exists(root_folder):
    os.makedirs(root_folder)

# Plot proximity vs. proximity weight
plt.figure(figsize=(8, 6))
plt.scatter(results_df['proximity_weight'], results_df['proximity'], c='blue', label='Proximity')
plt.title('Proximity vs. Proximity Weight')
plt.xlabel('Proximity Weight (λ₁)')
plt.ylabel('Proximity Metric')
plt.legend()
fig_file_path = os.path.join(root_folder, 'lambda_1_dice_x.eps')
plt.savefig(fig_file_path, format='eps')
plt.show()

# Plot diversity vs. diversity weight
plt.figure(figsize=(8, 6))
plt.scatter(results_df['diversity_weight'], results_df['diversity'], c='green', label='Diversity')
plt.title('Diversity vs. Diversity Weight')
plt.xlabel('Diversity Weight (λ₂)')
plt.ylabel('Diversity Metric')
plt.legend()
fig_file_path = os.path.join(root_folder, 'lambda_2_dice_x.eps')
plt.savefig(fig_file_path, format='eps')
plt.show()

# Plot robustness vs. robustness weight
plt.figure(figsize=(8, 6))
plt.scatter(results_df['robustness_weight'], results_df['robustness'], c='red', label='Robustness')
plt.title('Robustness vs. Robustness Weight')
plt.xlabel('Robustness Weight (λ₃)')
plt.ylabel('Robustness Metric')
plt.legend()
fig_file_path = os.path.join(root_folder, 'lambda_3_dice_x.eps')
plt.savefig(fig_file_path, format='eps')
plt.show()

As it can be seen from the charts proximity and diversity are not sensitive to changes in $\lambda_1$ and $\lambda_2$. On the other hand, robustness seems to be sensitive to changes in $\lambda_3$

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

cols_to_normalize = ['proximity', 'diversity', 'robustness']

results_normalized_df = results_df.copy().__deepcopy__()

results_normalized_df[cols_to_normalize] = scaler.fit_transform(results_normalized_df[cols_to_normalize])

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 15), subplot_kw={'projection': '3d'})

combinations = [
    ('proximity_weight', 'diversity_weight', 'proximity'),
    ('proximity_weight', 'robustness_weight', 'proximity'),
    ('diversity_weight', 'robustness_weight', 'proximity'),
    ('proximity_weight', 'diversity_weight', 'diversity'),
    ('proximity_weight', 'robustness_weight', 'diversity'),
    ('diversity_weight', 'robustness_weight', 'diversity'),
    ('proximity_weight', 'diversity_weight', 'robustness'),
    ('proximity_weight', 'robustness_weight', 'robustness'),
    ('diversity_weight', 'robustness_weight', 'robustness'),
]

titles = [
    'λ₁ vs λ₂ vs Proximity',
    'λ₁ vs λ₃ vs Proximity',
    'λ₂ vs λ₃ vs Proximity',
    'λ₁ vs λ₂ vs Diversity',
    'λ₁ vs λ₃ vs Diversity',
    'λ₂ vs λ₃ vs Diversity',
    'λ₁ vs λ₂ vs Robustness',
    'λ₁ vs λ₃ vs Robustness',
    'λ₂ vs λ₃ vs Robustness',
]

for ax, (x, y, z), title in zip(axes.flatten(), combinations, titles):
    sc = ax.scatter(results_normalized_df[x], results_normalized_df[y], results_normalized_df[z],
                    c=results_normalized_df[z], cmap='viridis')
    ax.set_xlabel(x)
    ax.set_ylabel(y)
    ax.set_zlabel(z)
    ax.set_title(title)
    fig.colorbar(sc, ax=ax, shrink=0.6, pad=0.1)

plt.tight_layout()
rood_folder = 'figure_artefacts'
if not os.path.exists(root_folder):
    os.makedirs(root_folder)
fig_file_path = os.path.join(root_folder, 'grid_search_dice_x.eps')
plt.savefig(fig_file_path, format='eps')
plt.show()

In [ ]:
results_df